# Module 4: Demo Chatbot + Output Quality Check

This notebook implements the reproducible demo layer of the final project. It wraps Module 2 and Module 3 evidence into a deterministic chatbot-style question-answering workflow and evaluates answer quality.

**Inputs:**  
- `module2_outputs/module2_llm_ready_patterns.json`  
- `module3_outputs/module3B_evidence_pack.json`  
- `module3_outputs/module3D_llm_analysis.json`  
- `module3_outputs/module3E_grounding_summary.json`

**Processing steps:**  
1. Build a unified demo context from upstream evidence files.  
2. Define fixed demo questions that cover the major project claims.  
3. Route each question into a query type.  
4. Retrieve relevant evidence for each routed question.  
5. Generate template-based grounded answers for reproducibility.  
6. Run answer-level quality checks for coverage, evidence use, and unsupported claims.  
7. Export a transcript for the report, presentation, and GitHub repository.

**Outputs:**  
- `module4_outputs/module4_demo_questions.json`  
- `module4_outputs/module4_query_log.csv`  
- `module4_outputs/module4_retrieved_evidence.json`  
- `module4_outputs/module4_chatbot_responses.json`  
- `module4_outputs/module4_answer_quality_summary.json`  
- `module4_outputs/module4_demo_transcript.md`

**Role in the full pipeline:**  
Module 4 demonstrates the final user-facing experience. It is intentionally deterministic so the demo remains reproducible even when API access is unavailable.

## Imports

**Input:** Standard Python runtime.

**Processing:** Import only standard-library modules needed for paths, JSON I/O, and CSV/log path definitions.

**Output:** Imported dependencies for the notebook skeleton.

In [1]:
# ### module4 demo chatbot quality check cell 2
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

from __future__ import annotations

import json
from pathlib import Path

## Constants and File Paths

**Input:** Known Module 2 and Module 3 output filenames, expected split values, classifiers, and normalization methods.

**Processing:** Define relative input paths, expected validation values, output directory, and expected Module 4 artifact paths.

**Output:** Reusable constants for validation and later Module 4 implementation cells.

In [2]:
# ### module4 demo chatbot quality check cell 4
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

MODULE2_LLM_READY_PATTERNS_PATH = Path("module2_outputs") / "module2_llm_ready_patterns.json"
MODULE2_SCENARIO_SUMMARY_PATH = Path("module2_outputs") / "module2_scenario_summary.json"
MODULE3B_EVIDENCE_PACK_PATH = Path("module3_outputs") / "module3B_evidence_pack.json"
MODULE3D_LLM_ANALYSIS_PATH = Path("module3_outputs") / "module3D_llm_analysis.json"
MODULE3E_GROUNDING_SUMMARY_PATH = Path("module3_outputs") / "module3E_grounding_summary.json"

REQUIRED_INPUT_FILES = [
    MODULE2_LLM_READY_PATTERNS_PATH,
    MODULE2_SCENARIO_SUMMARY_PATH,
    MODULE3B_EVIDENCE_PACK_PATH,
    MODULE3D_LLM_ANALYSIS_PATH,
    MODULE3E_GROUNDING_SUMMARY_PATH,
]

OUTPUT_DIR = Path("module4_outputs")

EXPECTED_SPLITS = [50, 70, 80, 90, 100]
EXPECTED_CLASSIFIERS = ["knn", "lasso", "pam", "rf", "svm", "xgb"]
EXPECTED_NORMALIZATIONS = ["mn", "non", "qn", "vsn"]

MODULE4_DEMO_CONTEXT_PATH = OUTPUT_DIR / "module4_demo_context.json"
MODULE4_DEMO_QUESTIONS_PATH = OUTPUT_DIR / "module4_demo_questions.json"
MODULE4_CHATBOT_RESPONSES_PATH = OUTPUT_DIR / "module4_chatbot_responses.json"
MODULE4_QUERY_LOG_PATH = OUTPUT_DIR / "module4_query_log.csv"
MODULE4_ANSWER_QUALITY_LOG_PATH = OUTPUT_DIR / "module4_answer_quality_log.csv"
MODULE4_ANSWER_QUALITY_SUMMARY_PATH = OUTPUT_DIR / "module4_answer_quality_summary.json"
MODULE4_DEMO_TRANSCRIPT_PATH = OUTPUT_DIR / "module4_demo_transcript.md"

## Helper Functions

**Input:** JSON-compatible Python dictionaries, relative file paths, and output directory paths.

**Processing:** Provide small reusable helpers for JSON loading, JSON saving, required input validation, and output directory creation.

**Output:** Typed utility functions for later Module 4 cells.

In [3]:
# ### module4 demo chatbot quality check cell 6
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: load_json
# Load a JSON artifact and raise a clear error if the file cannot be read.
def load_json(path: Path) -> dict:
    """Load a JSON file from a relative path.

    Input: Path to a JSON file.
    Output: Parsed JSON object as a dictionary.
    """
    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    if not isinstance(data, dict):
        raise TypeError(f"Expected JSON object in {path}, found {type(data).__name__}.")
    return data


# ### Function: save_json
# Write a Python object as formatted JSON for reproducible downstream use.
def save_json(obj: dict, path: Path) -> None:
    """Save a dictionary as formatted JSON.

    Input: JSON-compatible dictionary and target path.
    Output: Writes a UTF-8 JSON file and returns None.
    """
    with path.open("w", encoding="utf-8") as file:
        json.dump(obj, file, indent=2, ensure_ascii=False)


# ### Function: validate_input_files
# Verify that all required upstream files are available before execution.
def validate_input_files(paths: list[Path]) -> None:
    """Confirm that all required input files exist.

    Input: List of required relative file paths.
    Output: Raises FileNotFoundError if any file is missing; otherwise returns None.
    """
    missing_paths = [path for path in paths if not path.exists()]
    if missing_paths:
        missing_text = ", ".join(str(path) for path in missing_paths)
        raise FileNotFoundError(f"Missing required input file(s): {missing_text}")


# ### Function: make_output_dir
# Create the output directory if it does not already exist.
def make_output_dir(path: Path) -> None:
    """Create the Module 4 output directory if needed.

    Input: Directory path to create.
    Output: Ensures the directory exists and returns None.
    """
    path.mkdir(parents=True, exist_ok=True)

## Input Validation

**Input:** Required Module 2 and Module 3 JSON files plus expected split, classifier, and normalization definitions.

**Processing:** Create the Module 4 output directory, validate required inputs, load JSON data, and check scenario summary counts against expected values.

**Output:** Compact validation printout with scenario, classifier count, normalization count, curve count, and Module 3E grounding status.

In [4]:
# ### module4 demo chatbot quality check cell 8
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

make_output_dir(OUTPUT_DIR)
validate_input_files(REQUIRED_INPUT_FILES)

module2_patterns = load_json(MODULE2_LLM_READY_PATTERNS_PATH)
scenario_summary = load_json(MODULE2_SCENARIO_SUMMARY_PATH)
evidence_pack = load_json(MODULE3B_EVIDENCE_PACK_PATH)
llm_analysis = load_json(MODULE3D_LLM_ANALYSIS_PATH)
grounding_summary = load_json(MODULE3E_GROUNDING_SUMMARY_PATH)

if scenario_summary.get("split_values") != EXPECTED_SPLITS:
    raise ValueError("Unexpected split values in module2_scenario_summary.json.")

if scenario_summary.get("n_classifiers") != len(EXPECTED_CLASSIFIERS):
    raise ValueError("Unexpected classifier count in module2_scenario_summary.json.")

if scenario_summary.get("n_normalizations") != len(EXPECTED_NORMALIZATIONS):
    raise ValueError("Unexpected normalization count in module2_scenario_summary.json.")

expected_curve_count = len(EXPECTED_CLASSIFIERS) * len(EXPECTED_NORMALIZATIONS)
if scenario_summary.get("n_curves") != expected_curve_count:
    raise ValueError("Unexpected curve count in module2_scenario_summary.json.")

print("Module 4 validation summary")
print(f"scenario: {scenario_summary.get('scenario')}")
print(f"number of classifiers: {scenario_summary.get('n_classifiers')}")
print(f"number of normalizations: {scenario_summary.get('n_normalizations')}")
print(f"number of curves: {scenario_summary.get('n_curves')}")
print(f"Module 3E grounding status: {grounding_summary.get('overall_grounding_status')}")

Module 4 validation summary
scenario: jama_scenario1_3
number of classifiers: 6
number of normalizations: 4
number of curves: 24
Module 3E grounding status: pass_with_warnings


## Module 4A: Build Unified Demo Context

**Input:** Loaded Module 2 pattern evidence, Module 2 scenario summary, Module 3B evidence pack, Module 3D LLM analysis, and Module 3E grounding summary.

**Processing:** Build a compact grounded context for demo chatbot answers, including project constraints, scenario evidence, ranked combinations, analysis sections, grounding checks, and a per-combination lookup keyed as `classifier|normalization`.

**Output:** Saved `module4_outputs/module4_demo_context.json` plus a compact display of scenario metadata, grounding status, and top sensitive/stable combinations.

In [5]:
# ### module4 demo chatbot quality check cell 10
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: build_combination_lookup
# Index pattern records by classifier-normalization combination.
def build_combination_lookup(module2_patterns: dict) -> dict:
    """Build compact classifier-normalization evidence lookup.

    Input: Module 2 LLM-ready patterns dictionary with detected pattern entries.
    Output: Dictionary keyed by classifier|normalization with compact per-combination evidence.
    """
    detected_patterns = module2_patterns.get("detected_patterns", [])
    if not isinstance(detected_patterns, list):
        raise TypeError("module2_patterns['detected_patterns'] must be a list.")

    lookup = {}
    optional_fields = [
        "pattern_strength",
        "trend_label",
        "spike_type",
        "curve_shape",
        "relative_increase_pct",
        "max_jump",
        "max_jump_interval",
    ]

    for pattern in detected_patterns:
        classifier = pattern.get("classifier")
        normalization = pattern.get("normalization")
        if not classifier or not normalization:
            raise ValueError("Each detected pattern must include classifier and normalization.")

        error_curve = pattern.get("error_curve", {})
        key = f"{classifier}|{normalization}"
        entry = {
            "classifier": classifier,
            "normalization": normalization,
            "error_curve": error_curve,
            "delta_100_50": pattern.get("delta_100_50"),
            "robustness_flag": pattern.get("robustness_flag"),
            "degradation_type": pattern.get("degradation_type"),
            "pattern_sentence": pattern.get("pattern_sentence"),
            "error_50": error_curve.get("50"),
            "error_100": error_curve.get("100"),
        }
        for field in optional_fields:
            if field in pattern:
                entry[field] = pattern[field]
        lookup[key] = entry

    return lookup


# ### Function: build_demo_context
# Merge upstream outputs into one chatbot context object.
def build_demo_context(
    module2_patterns: dict,
    scenario_summary: dict,
    evidence_pack: dict,
    llm_analysis: dict,
    grounding_summary: dict,
) -> dict:
    """Build the compact grounded context used by the Module 4 demo chatbot.

    Input: Loaded dictionaries from Module 2 and Module 3 output JSON files.
    Output: Context dictionary containing constraints, evidence summaries, analysis text, grounding checks, and combination lookup.
    """
    combination_lookup = build_combination_lookup(module2_patterns)
    project_scope = evidence_pack.get("project_scope", {})
    explicit_constraints = list(project_scope.get("explicit_constraints", []))
    required_constraints = [
        "use only provided evidence",
        "do not infer biological mechanisms",
        "do not claim causality",
    ]

    for constraint in required_constraints:
        if constraint not in explicit_constraints:
            explicit_constraints.append(constraint)

    return {
        "module": "Module 4A Demo Chatbot Context",
        "chatbot_constraints": explicit_constraints,
        "project_scope": project_scope,
        "scenario_overview": evidence_pack.get("scenario_overview", {}),
        "scenario_summary": {
            "scenario": scenario_summary.get("scenario"),
            "metric": scenario_summary.get("metric"),
            "split_values": scenario_summary.get("split_values"),
            "n_classifiers": scenario_summary.get("n_classifiers"),
            "n_normalizations": scenario_summary.get("n_normalizations"),
            "n_curves": scenario_summary.get("n_curves"),
        },
        "most_sensitive_combinations": evidence_pack.get("most_sensitive_combinations", []),
        "most_stable_combinations": evidence_pack.get("most_stable_combinations", []),
        "classifier_level_evidence": evidence_pack.get("classifier_level_evidence", []),
        "normalization_level_evidence": evidence_pack.get("normalization_level_evidence", []),
        "representative_pattern_sentences": evidence_pack.get("representative_pattern_sentences", []),
        "llm_analysis_sections": llm_analysis,
        "grounding_summary": grounding_summary,
        "combination_lookup": combination_lookup,
    }


# ### Function: format_top_combinations
# Format the most important combinations for readable answers.
def format_top_combinations(combinations: list[dict], limit: int = 3) -> str:
    """Format ranked classifier-normalization combinations for compact display.

    Input: List of ranked combination dictionaries and maximum number to show.
    Output: Semicolon-separated display string using classifier|normalization and delta_100_50.
    """
    formatted = []
    for item in combinations[:limit]:
        key = f"{item.get('classifier')}|{item.get('normalization')}"
        formatted.append(f"{key}: {item.get('delta_100_50'):.3f}")
    return "; ".join(formatted)


module4_demo_context = build_demo_context(
    module2_patterns=module2_patterns,
    scenario_summary=scenario_summary,
    evidence_pack=evidence_pack,
    llm_analysis=llm_analysis,
    grounding_summary=grounding_summary,
)

combination_lookup = module4_demo_context["combination_lookup"]
required_lookup_fields = {
    "classifier",
    "normalization",
    "error_curve",
    "delta_100_50",
    "robustness_flag",
    "degradation_type",
    "pattern_sentence",
}

if len(combination_lookup) != 24:
    raise ValueError("combination_lookup must contain exactly 24 entries.")

for key, entry in combination_lookup.items():
    missing_fields = required_lookup_fields - set(entry)
    if missing_fields:
        raise ValueError(f"Combination {key} is missing required fields: {sorted(missing_fields)}")

if module4_demo_context["scenario_summary"].get("split_values") != EXPECTED_SPLITS:
    raise ValueError("Scenario split values do not match EXPECTED_SPLITS.")

represented_classifiers = sorted({entry["classifier"] for entry in combination_lookup.values()})
represented_normalizations = sorted({entry["normalization"] for entry in combination_lookup.values()})

if represented_classifiers != sorted(EXPECTED_CLASSIFIERS):
    raise ValueError("Not all expected classifiers are represented in combination_lookup.")

if represented_normalizations != sorted(EXPECTED_NORMALIZATIONS):
    raise ValueError("Not all expected normalizations are represented in combination_lookup.")

save_json(module4_demo_context, MODULE4_DEMO_CONTEXT_PATH)

print("Module 4A demo context summary")
print(f"scenario: {module4_demo_context['scenario_summary'].get('scenario')}")
print(f"metric: {module4_demo_context['scenario_summary'].get('metric')}")
print(f"split values: {module4_demo_context['scenario_summary'].get('split_values')}")
print(f"classifier-normalization combinations: {len(combination_lookup)}")
print(f"grounding status: {grounding_summary.get('overall_grounding_status')}")
print(f"top 3 sensitive: {format_top_combinations(module4_demo_context['most_sensitive_combinations'])}")
print(f"top 3 stable: {format_top_combinations(module4_demo_context['most_stable_combinations'])}")

Module 4A demo context summary
scenario: jama_scenario1_3
metric: classification error
split values: [50, 70, 80, 90, 100]
classifier-normalization combinations: 24
grounding status: pass_with_warnings
top 3 sensitive: knn|vsn: 0.306; knn|mn: 0.271; knn|qn: 0.270
top 3 stable: svm|non: 0.033; pam|qn: 0.050; rf|non: 0.059


## Module 4B: Define Demo Query Types and Fixed Test Questions

**Input:** `module4_outputs/module4_demo_context.json`.

**Processing:** Define fixed demo questions with expected query type, required evidence fields, expected entities, and answer requirements. Validate the question metadata against the Module 4A context.

**Output:** Saved `module4_outputs/module4_demo_questions.json` and a compact table of demo questions.

In [6]:
# ### module4 demo chatbot quality check cell 12
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

import sys

LOCAL_DEPS_PATH = Path(".pydeps")
if LOCAL_DEPS_PATH.exists() and str(LOCAL_DEPS_PATH) not in sys.path:
    sys.path.insert(0, str(LOCAL_DEPS_PATH))

try:
    import pandas as pd
    if not hasattr(pd, "DataFrame"):
        pd = None
except ImportError:
    pd = None


# ### Function: build_demo_questions
# Define fixed test questions that cover the main demo use cases.
def build_demo_questions() -> list[dict]:
    """Build the fixed Module 4B demo question set.

    Input: None; uses the fixed query specification for reproducible demo testing.
    Output: List of question dictionaries with expected routing and evidence metadata.
    """
    return [
        {
            "query_id": "Q1",
            "query": "What is the overall pattern observed across split values?",
            "expected_query_type": "scenario_overview",
            "required_evidence_fields": ["scenario_overview", "overall_pattern_counts", "dominant_observation", "split_values", "metric"],
            "expected_entities": {"classifiers": [], "normalizations": [], "splits": [50, 70, 80, 90, 100], "metric": "classification error"},
            "answer_requirements": ["mention classification error", "mention split values", "avoid causal claims", "include limitation language later in generated answer"],
        },
        {
            "query_id": "Q2",
            "query": "Which classifier appears most batch-sensitive?",
            "expected_query_type": "classifier_comparison",
            "required_evidence_fields": ["classifier_level_evidence", "most_sensitive_combinations", "overall_pattern_counts"],
            "expected_entities": {"classifiers": ["knn", "lasso", "pam", "rf", "svm", "xgb"], "normalizations": [], "splits": [50, 100], "metric": "classification error"},
            "answer_requirements": ["identify the most sensitive classifier", "cite classifier-level evidence", "mention top sensitive combinations", "avoid causal claims"],
        },
        {
            "query_id": "Q3",
            "query": "Which normalization method appears most stable across classifiers?",
            "expected_query_type": "normalization_comparison",
            "required_evidence_fields": ["normalization_level_evidence", "most_stable_combinations", "most_sensitive_combinations"],
            "expected_entities": {"classifiers": [], "normalizations": ["mn", "non", "qn", "vsn"], "splits": [50, 100], "metric": "classification error"},
            "answer_requirements": ["compare normalization methods", "identify the most stable normalization evidence", "mention classification error", "avoid causal claims"],
        },
        {
            "query_id": "Q4",
            "query": "Which classifier-normalization combination appears most stable?",
            "expected_query_type": "classifier_comparison",
            "required_evidence_fields": ["most_stable_combinations", "combination_lookup"],
            "expected_entities": {"classifiers": ["svm", "pam", "rf", "knn", "xgb"], "normalizations": ["non", "qn"], "splits": [50, 100], "metric": "classification error"},
            "answer_requirements": ["identify the most stable combination", "use delta_100_50", "mention error at split 50 and split 100 when available", "avoid causal claims"],
        },
        {
            "query_id": "Q5",
            "query": "How does split=100 differ from split=50 in this experiment?",
            "expected_query_type": "split_interpretation",
            "required_evidence_fields": ["split_values", "metric", "interpretation_note", "most_sensitive_combinations", "most_stable_combinations"],
            "expected_entities": {"classifiers": [], "normalizations": [], "splits": [50, 100], "metric": "classification error"},
            "answer_requirements": ["explain split 100 versus split 50", "mention stronger batch-separated evaluation", "use classification error", "avoid causal claims"],
        },
        {
            "query_id": "Q6",
            "query": "What does a large delta_100_50 indicate?",
            "expected_query_type": "split_interpretation",
            "required_evidence_fields": ["metric", "interpretation_note", "most_sensitive_combinations", "combination_lookup"],
            "expected_entities": {"classifiers": [], "normalizations": [], "splits": [50, 100], "metric": "classification error"},
            "answer_requirements": ["define delta_100_50", "connect large delta to higher classification error at split 100", "use examples from sensitive combinations", "avoid causal claims"],
        },
        {
            "query_id": "Q7",
            "query": "How should I interpret knn with vsn normalization?",
            "expected_query_type": "specific_combination",
            "required_evidence_fields": ["combination_lookup"],
            "expected_entities": {"classifiers": ["knn"], "normalizations": ["vsn"], "splits": [50, 100], "metric": "classification error"},
            "expected_combination_key": "knn|vsn",
            "answer_requirements": ["use knn|vsn lookup evidence", "mention error curve or split 50 and split 100 errors", "mention robustness flag and degradation type", "avoid causal claims"],
        },
        {
            "query_id": "Q8",
            "query": "Are the LLM-generated interpretations consistent with the observed evidence?",
            "expected_query_type": "grounding_reliability",
            "required_evidence_fields": ["grounding_summary", "llm_analysis", "project_constraints"],
            "expected_entities": {"classifiers": [], "normalizations": [], "splits": [], "metric": "classification error"},
            "answer_requirements": ["report grounding status", "mention pass_with_warnings when applicable", "describe consistency using grounding evidence", "avoid overstating reliability"],
        },
        {
            "query_id": "Q9",
            "query": "What are the main limitations of this analysis?",
            "expected_query_type": "limitation_summary",
            "required_evidence_fields": ["project_constraints", "analysis_scope_note", "grounding_summary", "llm_analysis"],
            "expected_entities": {"classifiers": [], "normalizations": [], "splits": [], "metric": "classification error"},
            "answer_requirements": ["mention evidence scope", "mention no biological mechanism inference", "mention no causal claims", "avoid adding external evidence"],
        },
        {
            "query_id": "Q10",
            "query": "Can you summarize the result in a report-ready paragraph?",
            "expected_query_type": "report_ready_summary",
            "required_evidence_fields": ["scenario_overview", "dominant_observation", "most_sensitive_combinations", "most_stable_combinations", "classifier_level_evidence", "normalization_level_evidence", "project_constraints"],
            "expected_entities": {"classifiers": ["knn", "lasso", "pam", "rf", "svm", "xgb"], "normalizations": ["mn", "non", "qn", "vsn"], "splits": [50, 100], "metric": "classification error"},
            "answer_requirements": ["write one report-ready paragraph", "mention dominant pattern", "include sensitive and stable examples", "include limitations and constraints"],
        },
    ]


# ### Function: save_demo_questions
# Save the fixed demo-question set for reproducibility.
def save_demo_questions(questions: list[dict], output_path: Path) -> None:
    """Save fixed demo questions as formatted JSON.

    Input: List of question dictionaries and target JSON path.
    Output: Writes the question list to disk and returns None.
    """
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(questions, file, indent=2, ensure_ascii=False)


# ### Function: validate_demo_questions
# Check whether each demo question has a supported type and available evidence.
def validate_demo_questions(questions: list[dict], context: dict) -> None:
    """Validate demo question schema and expected entities against context.

    Input: Fixed demo question list and loaded Module 4A context dictionary.
    Output: Raises ValueError for invalid metadata; otherwise returns None.
    """
    required_query_types = {
        "scenario_overview",
        "classifier_comparison",
        "normalization_comparison",
        "specific_combination",
        "split_interpretation",
        "grounding_reliability",
        "limitation_summary",
        "report_ready_summary",
    }
    required_fields = {
        "query_id",
        "query",
        "expected_query_type",
        "required_evidence_fields",
        "expected_entities",
        "answer_requirements",
    }

    if len(questions) != 10:
        raise ValueError("There must be exactly 10 demo questions.")

    expected_ids = [f"Q{index}" for index in range(1, 11)]
    actual_ids = [question.get("query_id") for question in questions]
    if actual_ids != expected_ids:
        raise ValueError("query_id values must be Q1 through Q10 in order.")

    actual_query_types = {question.get("expected_query_type") for question in questions}
    if not required_query_types.issubset(actual_query_types):
        missing_types = sorted(required_query_types - actual_query_types)
        raise ValueError(f"Missing required query type(s): {missing_types}")

    lookup = context.get("combination_lookup", {})
    for question in questions:
        missing_fields = required_fields - set(question)
        if missing_fields:
            raise ValueError(f"{question.get('query_id')} is missing fields: {sorted(missing_fields)}")

        combination_key = question.get("expected_combination_key")
        if combination_key and combination_key not in lookup:
            raise ValueError(f"Unknown expected combination key: {combination_key}")

        entities = question["expected_entities"]
        for classifier in entities.get("classifiers", []):
            if classifier not in EXPECTED_CLASSIFIERS:
                raise ValueError(f"Unknown expected classifier: {classifier}")
        for normalization in entities.get("normalizations", []):
            if normalization not in EXPECTED_NORMALIZATIONS:
                raise ValueError(f"Unknown expected normalization: {normalization}")
        for split in entities.get("splits", []):
            if split not in EXPECTED_SPLITS:
                raise ValueError(f"Unknown expected split: {split}")

    specific_questions = [question for question in questions if question["expected_query_type"] == "specific_combination"]
    if not any(question.get("expected_combination_key") == "knn|vsn" for question in specific_questions):
        raise ValueError("The specific_combination question must include expected_combination_key='knn|vsn'.")


module4_context_for_questions = load_json(MODULE4_DEMO_CONTEXT_PATH)
module4_demo_questions = build_demo_questions()
validate_demo_questions(module4_demo_questions, module4_context_for_questions)
save_demo_questions(module4_demo_questions, MODULE4_DEMO_QUESTIONS_PATH)

questions_display_rows = [
    {
        "query_id": question["query_id"],
        "expected_query_type": question["expected_query_type"],
        "query": question["query"],
        "required_evidence_fields": ", ".join(question["required_evidence_fields"]),
    }
    for question in module4_demo_questions
]
questions_display = pd.DataFrame(questions_display_rows) if pd is not None else questions_display_rows
questions_display

,query_id,expected_query_type,query,required_evidence_fields
0,Q1,scenario_overview,What is the overall pattern observed across sp...,"scenario_overview, overall_pattern_counts, dom..."
1,Q2,classifier_comparison,Which classifier appears most batch-sensitive?,"classifier_level_evidence, most_sensitive_comb..."
2,Q3,normalization_comparison,Which normalization method appears most stable...,"normalization_level_evidence, most_stable_comb..."
3,Q4,classifier_comparison,Which classifier-normalization combination app...,"most_stable_combinations, combination_lookup"
4,Q5,split_interpretation,How does split=100 differ from split=50 in thi...,"split_values, metric, interpretation_note, mos..."
5,Q6,split_interpretation,What does a large delta_100_50 indicate?,"metric, interpretation_note, most_sensitive_co..."
6,Q7,specific_combination,How should I interpret knn with vsn normalizat...,combination_lookup
7,Q8,grounding_reliability,Are the LLM-generated interpretations consiste...,"grounding_summary, llm_analysis, project_const..."
8,Q9,limitation_summary,What are the main limitations of this analysis?,"project_constraints, analysis_scope_note, grou..."
9,Q10,report_ready_summary,Can you summarize the result in a report-ready...,"scenario_overview, dominant_observation, most_..."


## Module 4C: Lightweight Rule-Based Query Router

**Input:** `module4_outputs/module4_demo_context.json` and `module4_outputs/module4_demo_questions.json`.

**Processing:** Route each fixed demo question with deterministic priority rules, extract classifiers, normalizations, split values, and combination keys, then assign route subtypes for later evidence retrieval.

**Output:** Saved `module4_outputs/module4_query_log.csv` plus a compact routing table and validation summary.

In [7]:
# ### module4 demo chatbot quality check cell 14
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

import csv
import re


# ### Function: normalize_query_text
# Normalize query text before routing and retrieval.
def normalize_query_text(query: str) -> str:
    """Normalize query text for deterministic rule matching.

    Input: Raw user-facing query string.
    Output: Lowercased query with normalized punctuation and spaces.
    """
    query_norm = query.lower().replace("-", " ")
    query_norm = re.sub(r"[^a-z0-9_=\s]", " ", query_norm)
    return re.sub(r"\s+", " ", query_norm).strip()


# ### Function: extract_query_entities
# Identify classifiers, normalizations, and split values mentioned in a query.
def extract_query_entities(query: str, context: dict) -> dict:
    """Extract supported classifiers, normalizations, splits, and combination keys.

    Input: Raw query string and Module 4A context dictionary.
    Output: Entity dictionary with mentioned names, split values, expected combination key, and key-existence flag.
    """
    query_norm = normalize_query_text(query)
    lookup = context.get("combination_lookup", {})
    supported_classifiers = sorted({entry["classifier"] for entry in lookup.values()})
    supported_normalizations = sorted({entry["normalization"] for entry in lookup.values()})
    supported_splits = context.get("scenario_summary", {}).get("split_values", EXPECTED_SPLITS)

    classifiers = [name for name in supported_classifiers if re.search(rf"\b{re.escape(name)}\b", query_norm)]
    normalizations = [name for name in supported_normalizations if re.search(rf"\b{re.escape(name)}\b", query_norm)]

    split_values = []
    split_context_present = "split" in query_norm or "delta_100_50" in query_norm
    for split in supported_splits:
        split_text = str(split)
        explicit_split = re.search(rf"\bsplit\s*=?\s*{split_text}\b", query_norm)
        bare_split = split_context_present and re.search(rf"\b{split_text}\b", query_norm)
        if explicit_split or bare_split:
            split_values.append(split)

    expected_combination_key = ""
    combination_key_exists = False
    if classifiers and normalizations:
        expected_combination_key = f"{classifiers[0]}|{normalizations[0]}"
        combination_key_exists = expected_combination_key in lookup

    return {
        "classifier_mentioned": classifiers,
        "normalization_mentioned": normalizations,
        "split_values_mentioned": split_values,
        "expected_combination_key": expected_combination_key,
        "combination_key_exists": combination_key_exists,
    }


# ### Function: infer_route_subtype
# Refine the broad query type into a more specific route.
def infer_route_subtype(query_norm: str, detected_query_type: str, entities: dict) -> str:
    """Infer the route subtype for targeted downstream evidence retrieval.

    Input: Normalized query text, detected top-level query type, and extracted entity dictionary.
    Output: Route subtype string.
    """
    if detected_query_type == "report_ready_summary":
        return "report_ready_summary"
    if detected_query_type == "limitation_summary":
        return "limitations"
    if detected_query_type == "grounding_reliability":
        return "grounding_consistency"
    if detected_query_type == "specific_combination":
        return "specific_combination"
    if detected_query_type == "split_interpretation":
        if "delta_100_50" in query_norm or "large delta" in query_norm or re.search(r"\bdelta\b", query_norm):
            return "delta_interpretation"
        return "split_50_vs_100"
    if detected_query_type == "normalization_comparison":
        return "stable_normalization"
    if detected_query_type == "classifier_comparison":
        if "combination" in query_norm or "classifier normalization" in query_norm:
            return "stable_combination_comparison"
        return "batch_sensitive_classifier"
    if detected_query_type == "scenario_overview":
        return "overall_pattern"
    return "unknown"


# ### Function: infer_retrieved_context_keys
# Select which context fields should be retrieved for a routed query.
def infer_retrieved_context_keys(detected_query_type: str, route_subtype: str) -> list[str]:
    """Map a route subtype to compact context keys for later retrieval.

    Input: Detected top-level query type and route subtype.
    Output: List of context key labels expected by downstream evidence retrieval.
    """
    route_context_keys = {
        "overall_pattern": ["scenario_overview", "overall_pattern_counts", "dominant_observation", "split_values", "metric"],
        "batch_sensitive_classifier": ["classifier_level_evidence", "most_sensitive_combinations", "overall_pattern_counts"],
        "stable_normalization": ["normalization_level_evidence", "most_stable_combinations", "most_sensitive_combinations"],
        "stable_combination_comparison": ["most_stable_combinations", "combination_lookup"],
        "split_50_vs_100": ["split_values", "metric", "interpretation_note", "most_sensitive_combinations", "most_stable_combinations"],
        "delta_interpretation": ["metric", "interpretation_note", "most_sensitive_combinations", "combination_lookup"],
        "specific_combination": ["combination_lookup"],
        "grounding_consistency": ["grounding_summary", "llm_analysis", "project_constraints"],
        "limitations": ["project_constraints", "analysis_scope_note", "grounding_summary", "llm_analysis"],
        "report_ready_summary": ["scenario_overview", "dominant_observation", "most_sensitive_combinations", "most_stable_combinations", "classifier_level_evidence", "normalization_level_evidence", "project_constraints"],
    }
    return route_context_keys.get(route_subtype, [])


# ### Function: route_query
# Route one question to a query type and retrieval plan.
def route_query(query: str, context: dict) -> dict:
    """Route one query and return its detected type, subtype, entities, and retrieval keys.

    Input: Raw query string and Module 4A context dictionary.
    Output: Routed query metadata dictionary.
    """
    query_norm = normalize_query_text(query)
    entities = extract_query_entities(query, context)

    if any(phrase in query_norm for phrase in ["report ready", "paragraph", "summarize the result", "summary paragraph"]):
        detected_query_type = "report_ready_summary"
    elif any(phrase in query_norm for phrase in ["limitation", "limitations", "caution", "constraints", "main limitations"]):
        detected_query_type = "limitation_summary"
    elif any(phrase in query_norm for phrase in ["consistent", "consistency", "grounding", "reliable", "reliability", "observed evidence", "llm generated"]):
        detected_query_type = "grounding_reliability"
    elif entities["classifier_mentioned"] and entities["normalization_mentioned"]:
        detected_query_type = "specific_combination"
    elif any(phrase in query_norm for phrase in ["split=100", "split 100", "split=50", "split 50", "delta_100_50", "large delta"] ) or re.search(r"\bdelta\b", query_norm):
        detected_query_type = "split_interpretation"
    elif (
        "combination" not in query_norm
        and "classifier normalization" not in query_norm
        and any(phrase in query_norm for phrase in ["normalization", "normalization method", "normalization methods", "stable across classifiers"])
        and any(phrase in query_norm for phrase in ["stable", "comparison", "compare", "method", "methods"])
    ):
        detected_query_type = "normalization_comparison"
    elif any(phrase in query_norm for phrase in ["classifier", "batch sensitive", "sensitive", "most stable", "classifier normalization combination", "combination"]):
        detected_query_type = "classifier_comparison"
    elif any(phrase in query_norm for phrase in ["overall", "overall pattern", "pattern observed", "across split", "across split values", "observed across"]):
        detected_query_type = "scenario_overview"
    else:
        detected_query_type = "unknown"

    route_subtype = infer_route_subtype(query_norm, detected_query_type, entities)
    retrieved_context_keys = infer_retrieved_context_keys(detected_query_type, route_subtype)

    return {
        "detected_query_type": detected_query_type,
        "route_subtype": route_subtype,
        "classifier_mentioned": entities["classifier_mentioned"],
        "normalization_mentioned": entities["normalization_mentioned"],
        "split_values_mentioned": entities["split_values_mentioned"],
        "expected_combination_key": entities["expected_combination_key"],
        "combination_key_exists": entities["combination_key_exists"],
        "retrieved_context_keys": retrieved_context_keys,
    }


# ### Function: route_demo_questions
# Route all fixed demo questions and return a query log table.
def route_demo_questions(questions: list[dict], context: dict) -> "pd.DataFrame":
    """Route all fixed demo questions and compare detected and expected query types.

    Input: Fixed demo question dictionaries and Module 4A context dictionary.
    Output: DataFrame-like route log with one row per demo question.
    """
    rows = []
    for question in questions:
        routed = route_query(question["query"], context)
        rows.append(
            {
                "query_id": question["query_id"],
                "query": question["query"],
                "expected_query_type": question["expected_query_type"],
                **routed,
                "route_match": routed["detected_query_type"] == question["expected_query_type"],
            }
        )
    return pd.DataFrame(rows) if pd is not None else rows


# ### Function: _table_to_rows
# Convert a table-like object into dictionaries for saving.
def _table_to_rows(table: "pd.DataFrame") -> list[dict]:
    """Convert a DataFrame-like table to row dictionaries.

    Input: pandas DataFrame or list of row dictionaries.
    Output: List of row dictionaries.
    """
    if pd is not None and hasattr(table, "to_dict"):
        return table.to_dict(orient="records")
    return list(table)


# ### Function: save_query_log
# Save the query-routing table for auditability.
def save_query_log(query_log_df: "pd.DataFrame", output_path: Path) -> None:
    """Save routed query records to CSV with compact JSON list columns.

    Input: DataFrame-like routed query log and target CSV path.
    Output: Writes CSV file and returns None.
    """
    rows = _table_to_rows(query_log_df)
    fieldnames = [
        "query_id",
        "query",
        "expected_query_type",
        "detected_query_type",
        "route_subtype",
        "classifier_mentioned",
        "normalization_mentioned",
        "split_values_mentioned",
        "expected_combination_key",
        "combination_key_exists",
        "route_match",
        "retrieved_context_keys",
    ]
    try:
        with output_path.open("w", encoding="utf-8", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            for row in rows:
                csv_row = row.copy()
                for column in ["classifier_mentioned", "normalization_mentioned", "split_values_mentioned", "retrieved_context_keys"]:
                    csv_row[column] = json.dumps(csv_row[column], ensure_ascii=False, separators=(",", ":"))
                writer.writerow(csv_row)
    except PermissionError:
        if not output_path.exists():
            raise


module4_context_for_routing = load_json(MODULE4_DEMO_CONTEXT_PATH)
with MODULE4_DEMO_QUESTIONS_PATH.open("r", encoding="utf-8") as file:
    module4_questions_for_routing = json.load(file)
module4_query_log = route_demo_questions(module4_questions_for_routing, module4_context_for_routing)
query_log_rows = _table_to_rows(module4_query_log)

expected_route_ids = [f"Q{index}" for index in range(1, 11)]
actual_route_ids = [row["query_id"] for row in query_log_rows]
if len(query_log_rows) != 10:
    raise ValueError("There must be exactly 10 routed rows.")
if actual_route_ids != expected_route_ids:
    raise ValueError("Routed query IDs must be Q1 through Q10 in order.")
if not all(row["route_match"] for row in query_log_rows):
    raise ValueError("All fixed demo questions must have matching detected and expected query types.")
if any(row["detected_query_type"] == "unknown" for row in query_log_rows):
    raise ValueError("No fixed demo question should route to unknown.")

row_by_id = {row["query_id"]: row for row in query_log_rows}
if row_by_id["Q7"]["classifier_mentioned"] != ["knn"]:
    raise ValueError("Q7 must mention classifier knn.")
if row_by_id["Q7"]["normalization_mentioned"] != ["vsn"]:
    raise ValueError("Q7 must mention normalization vsn.")
if row_by_id["Q7"]["expected_combination_key"] != "knn|vsn" or not row_by_id["Q7"]["combination_key_exists"]:
    raise ValueError("Q7 must resolve to existing combination key knn|vsn.")
if sorted(row_by_id["Q5"]["split_values_mentioned"]) != [50, 100]:
    raise ValueError("Q5 must include split values 50 and 100.")
if row_by_id["Q6"]["route_subtype"] != "delta_interpretation":
    raise ValueError("Q6 must route to delta_interpretation.")
if row_by_id["Q8"]["route_subtype"] != "grounding_consistency":
    raise ValueError("Q8 must route to grounding_consistency.")
if row_by_id["Q9"]["route_subtype"] != "limitations":
    raise ValueError("Q9 must route to limitations.")
if row_by_id["Q10"]["route_subtype"] != "report_ready_summary":
    raise ValueError("Q10 must route to report_ready_summary.")

save_query_log(module4_query_log, MODULE4_QUERY_LOG_PATH)

display_columns = [
    "query_id",
    "expected_query_type",
    "detected_query_type",
    "route_subtype",
    "classifier_mentioned",
    "normalization_mentioned",
    "split_values_mentioned",
    "route_match",
]
if pd is not None:
    route_display = module4_query_log[display_columns]
else:
    route_display = [{column: row[column] for column in display_columns} for row in query_log_rows]

route_matches = sum(1 for row in query_log_rows if row["route_match"])
unknown_routes = sum(1 for row in query_log_rows if row["detected_query_type"] == "unknown")
mismatched_query_ids = [row["query_id"] for row in query_log_rows if not row["route_match"]]

print("Module 4C routing validation summary")
print(f"number of routed queries: {len(query_log_rows)}")
print(f"number of route matches: {route_matches}")
print(f"number of unknown routes: {unknown_routes}")
print(f"mismatched query IDs: {mismatched_query_ids}")
route_display

Module 4C routing validation summary
number of routed queries: 10
number of route matches: 10
number of unknown routes: 0
mismatched query IDs: []


,query_id,expected_query_type,detected_query_type,route_subtype,classifier_mentioned,normalization_mentioned,split_values_mentioned,route_match
0,Q1,scenario_overview,scenario_overview,overall_pattern,[],[],[],True
1,Q2,classifier_comparison,classifier_comparison,batch_sensitive_classifier,[],[],[],True
2,Q3,normalization_comparison,normalization_comparison,stable_normalization,[],[],[],True
3,Q4,classifier_comparison,classifier_comparison,stable_combination_comparison,[],[],[],True
4,Q5,split_interpretation,split_interpretation,split_50_vs_100,[],[],"[50, 100]",True
5,Q6,split_interpretation,split_interpretation,delta_interpretation,[],[],[],True
6,Q7,specific_combination,specific_combination,specific_combination,[knn],[vsn],[],True
7,Q8,grounding_reliability,grounding_reliability,grounding_consistency,[],[],[],True
8,Q9,limitation_summary,limitation_summary,limitations,[],[],[],True
9,Q10,report_ready_summary,report_ready_summary,report_ready_summary,[],[],[],True


## Module 4D: Evidence Retrieval for Routed Demo Queries

**Input:** `module4_outputs/module4_demo_context.json`, `module4_outputs/module4_demo_questions.json`, and `module4_outputs/module4_query_log.csv`.

**Processing:** Retrieve compact evidence for each routed query by `route_subtype`, keeping only the context fields needed by later grounded answer generation.

**Output:** Saved `module4_outputs/module4_retrieved_evidence.json` plus a compact retrieval table and validation summary.

In [8]:
# ### module4 demo chatbot quality check cell 16
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

import ast


# ### Function: parse_list_cell
# Parse list-like CSV cells back into Python lists.
def parse_list_cell(value) -> list:
    """Parse a CSV cell that may contain a list-like value.

    Input: CSV cell value, Python list, or missing value.
    Output: Parsed Python list; returns an empty list when parsing is not possible.
    """
    if isinstance(value, list):
        return value
    if value is None or value == "":
        return []
    if not isinstance(value, str):
        return []
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError:
        try:
            parsed = ast.literal_eval(value)
        except (SyntaxError, ValueError):
            return []
    return parsed if isinstance(parsed, list) else []


# ### Function: get_context_value
# Safely retrieve a value from the demo context.
def get_context_value(context: dict, key: str, default=None):
    """Safely read a top-level context value.

    Input: Context dictionary, key, and optional default value.
    Output: Context value if present; otherwise the default.
    """
    return context.get(key, default)


# ### Function: compact_combination_entry
# Reduce one combination record to the fields needed for an answer.
def compact_combination_entry(entry: dict) -> dict:
    """Keep only answer-generation fields from one combination entry.

    Input: Full per-combination evidence dictionary.
    Output: Compact combination evidence dictionary.
    """
    fields = [
        "classifier",
        "normalization",
        "error_curve",
        "error_50",
        "error_100",
        "delta_100_50",
        "relative_increase_pct",
        "trend_label",
        "robustness_flag",
        "spike_type",
        "curve_shape",
        "pattern_strength",
        "degradation_type",
        "pattern_sentence",
    ]
    return {field: entry.get(field) for field in fields if field in entry}


# ### Function: get_combination_by_key
# Retrieve one classifier-normalization combination by key.
def get_combination_by_key(context: dict, key: str) -> dict:
    """Retrieve and compact one exact classifier-normalization combination.

    Input: Module 4A context dictionary and combination key such as knn|vsn.
    Output: Compact combination evidence dictionary, or an empty dictionary if missing.
    """
    entry = context.get("combination_lookup", {}).get(key, {})
    return compact_combination_entry(entry) if entry else {}


# ### Function: get_stable_combination_lookup_subset
# Select stable combinations for comparison answers.
def get_stable_combination_lookup_subset(context: dict) -> dict:
    """Retrieve compact lookup entries for the stable combination list only.

    Input: Module 4A context dictionary.
    Output: Dictionary of compact combination evidence keyed by classifier|normalization.
    """
    subset = {}
    for item in context.get("most_stable_combinations", []):
        key = f"{item.get('classifier')}|{item.get('normalization')}"
        compact_entry = get_combination_by_key(context, key)
        if compact_entry:
            subset[key] = compact_entry
    return subset


# ### Function: _context_aliases
# Define aliases for context keys used during retrieval.
def _context_aliases(context: dict) -> dict:
    """Build commonly used context alias values.

    Input: Module 4A context dictionary.
    Output: Dictionary of normalized alias values for retrieval payloads.
    """
    project_scope = context.get("project_scope", {})
    scenario_summary = context.get("scenario_summary", {})
    scenario_overview = context.get("scenario_overview", {})
    return {
        "scenario": scenario_summary.get("scenario", project_scope.get("scenario")),
        "metric": scenario_summary.get("metric", project_scope.get("metric")),
        "split_values": scenario_summary.get("split_values", project_scope.get("split_values")),
        "interpretation_note": project_scope.get("interpretation_note"),
        "analysis_scope_note": project_scope.get("analysis_scope_note"),
        "project_constraints": context.get("chatbot_constraints", []),
        "overall_pattern_counts": scenario_overview.get("overall_pattern_counts", {}),
        "dominant_observation": scenario_overview.get("dominant_observation"),
        "llm_analysis": context.get("llm_analysis_sections", {}),
    }


# ### Function: retrieve_evidence_for_query
# Retrieve evidence for one routed query.
def retrieve_evidence_for_query(route_record: dict, context: dict) -> dict:
    """Retrieve compact evidence for one routed query.

    Input: One routed query record and Module 4A context dictionary.
    Output: Retrieved evidence record with status and warnings.
    """
    aliases = _context_aliases(context)
    route_subtype = route_record.get("route_subtype") or "unknown"
    if route_subtype == "unknown":
        route_subtype = route_record.get("detected_query_type", "unknown")

    evidence = {}
    warnings = []

    if route_subtype == "overall_pattern":
        evidence = {
            "scenario": aliases["scenario"],
            "metric": aliases["metric"],
            "split_values": aliases["split_values"],
            "scenario_overview": context.get("scenario_overview", {}),
            "overall_pattern_counts": aliases["overall_pattern_counts"],
            "dominant_observation": aliases["dominant_observation"],
            "analysis_scope_note": aliases["analysis_scope_note"],
            "project_constraints": aliases["project_constraints"],
        }
    elif route_subtype == "batch_sensitive_classifier":
        evidence = {
            "metric": aliases["metric"],
            "split_values": aliases["split_values"],
            "classifier_level_evidence": context.get("classifier_level_evidence", []),
            "most_sensitive_combinations": context.get("most_sensitive_combinations", []),
            "most_stable_combinations": context.get("most_stable_combinations", []),
            "overall_pattern_counts": aliases["overall_pattern_counts"],
            "dominant_observation": aliases["dominant_observation"],
        }
    elif route_subtype == "stable_normalization":
        evidence = {
            "metric": aliases["metric"],
            "split_values": aliases["split_values"],
            "normalization_level_evidence": context.get("normalization_level_evidence", []),
            "most_stable_combinations": context.get("most_stable_combinations", []),
            "most_sensitive_combinations": context.get("most_sensitive_combinations", []),
            "analysis_scope_note": aliases["analysis_scope_note"],
        }
    elif route_subtype == "stable_combination_comparison":
        evidence = {
            "metric": aliases["metric"],
            "split_values": aliases["split_values"],
            "most_stable_combinations": context.get("most_stable_combinations", []),
            "most_sensitive_combinations": context.get("most_sensitive_combinations", []),
            "stable_combination_lookup_subset": get_stable_combination_lookup_subset(context),
        }
    elif route_subtype == "split_50_vs_100":
        evidence = {
            "metric": aliases["metric"],
            "split_values": aliases["split_values"],
            "interpretation_note": aliases["interpretation_note"],
            "dominant_observation": aliases["dominant_observation"],
            "most_sensitive_combinations": context.get("most_sensitive_combinations", []),
            "most_stable_combinations": context.get("most_stable_combinations", []),
            "project_constraints": aliases["project_constraints"],
        }
    elif route_subtype == "delta_interpretation":
        sensitive = context.get("most_sensitive_combinations", [])
        stable = context.get("most_stable_combinations", [])
        evidence = {
            "metric": aliases["metric"],
            "split_values": aliases["split_values"],
            "interpretation_note": aliases["interpretation_note"],
            "most_sensitive_combinations": sensitive,
            "most_stable_combinations": stable,
            "example_high_delta_combination": sensitive[0] if sensitive else {},
            "example_low_delta_combination": stable[0] if stable else {},
            "project_constraints": aliases["project_constraints"],
        }
    elif route_subtype == "specific_combination":
        key = route_record.get("expected_combination_key", "")
        combination_evidence = get_combination_by_key(context, key)
        evidence = {
            "metric": aliases["metric"],
            "split_values": aliases["split_values"],
            "expected_combination_key": key,
            "combination_key_exists": bool(combination_evidence),
            "combination_evidence": combination_evidence,
            "project_constraints": aliases["project_constraints"],
        }
        if not combination_evidence:
            warnings.append(f"No combination evidence found for key {key}.")
    elif route_subtype == "grounding_consistency":
        grounding_summary = context.get("grounding_summary", {})
        llm_analysis = aliases["llm_analysis"]
        evidence = {
            "grounding_summary": {
                "overall_grounding_status": grounding_summary.get("overall_grounding_status"),
                "required_sections_passed": grounding_summary.get("required_sections_passed"),
                "classifier_coverage_rate": grounding_summary.get("classifier_coverage_rate"),
                "normalization_coverage_rate": grounding_summary.get("normalization_coverage_rate"),
                "top_sensitive_pair_coverage": grounding_summary.get("top_sensitive_pair_coverage"),
                "top_stable_pair_coverage": grounding_summary.get("top_stable_pair_coverage"),
            },
            "llm_analysis": {"evidence_usage_notes": llm_analysis.get("evidence_usage_notes", [])},
            "project_constraints": aliases["project_constraints"],
            "analysis_scope_note": aliases["analysis_scope_note"],
        }
    elif route_subtype == "limitations":
        grounding_summary = context.get("grounding_summary", {})
        llm_analysis = aliases["llm_analysis"]
        evidence = {
            "project_constraints": aliases["project_constraints"],
            "analysis_scope_note": aliases["analysis_scope_note"],
            "grounding_summary": {
                "overall_grounding_status": grounding_summary.get("overall_grounding_status"),
                "forbidden_terms_found": grounding_summary.get("forbidden_terms_found", []),
                "classifier_coverage_rate": grounding_summary.get("classifier_coverage_rate"),
                "normalization_coverage_rate": grounding_summary.get("normalization_coverage_rate"),
            },
            "llm_analysis_limitations": llm_analysis.get("limitations"),
        }
    elif route_subtype == "report_ready_summary":
        evidence = {
            "scenario": aliases["scenario"],
            "metric": aliases["metric"],
            "split_values": aliases["split_values"],
            "scenario_overview": context.get("scenario_overview", {}),
            "dominant_observation": aliases["dominant_observation"],
            "most_sensitive_combinations": context.get("most_sensitive_combinations", []),
            "most_stable_combinations": context.get("most_stable_combinations", []),
            "classifier_level_evidence": context.get("classifier_level_evidence", []),
            "normalization_level_evidence": context.get("normalization_level_evidence", []),
            "grounding_summary": context.get("grounding_summary", {}),
            "project_constraints": aliases["project_constraints"],
            "analysis_scope_note": aliases["analysis_scope_note"],
        }
    else:
        warnings.append(f"Unsupported route subtype: {route_subtype}")

    if not evidence:
        status = "failed"
    elif warnings:
        status = "success_with_warnings"
    else:
        status = "success"

    return {
        "query_id": route_record.get("query_id"),
        "query": route_record.get("query"),
        "detected_query_type": route_record.get("detected_query_type"),
        "route_subtype": route_record.get("route_subtype"),
        "retrieved_context_keys": parse_list_cell(route_record.get("retrieved_context_keys")),
        "retrieved_evidence": evidence,
        "retrieval_status": status,
        "retrieval_warnings": warnings,
    }


# ### Function: retrieve_evidence_for_demo_queries
# Retrieve evidence for all demo questions.
def retrieve_evidence_for_demo_queries(query_log_df: "pd.DataFrame", context: dict) -> list[dict]:
    """Retrieve evidence for all routed fixed demo questions.

    Input: Routed query log table and Module 4A context dictionary.
    Output: List of retrieved evidence records.
    """
    return [retrieve_evidence_for_query(row, context) for row in _table_to_rows(query_log_df)]


# ### Function: save_retrieved_evidence
# Save retrieved evidence records for reproducibility.
def save_retrieved_evidence(records: list[dict], output_path: Path) -> None:
    """Save retrieved evidence records as formatted JSON.

    Input: Retrieved evidence records and target JSON path.
    Output: Writes JSON file and returns None.
    """
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(records, file, indent=2, ensure_ascii=False)


module4_context_for_retrieval = load_json(MODULE4_DEMO_CONTEXT_PATH)
with MODULE4_DEMO_QUESTIONS_PATH.open("r", encoding="utf-8") as file:
    module4_questions_for_retrieval = json.load(file)
with MODULE4_QUERY_LOG_PATH.open("r", encoding="utf-8", newline="") as file:
    query_log_rows_for_retrieval = list(csv.DictReader(file))

if len(module4_questions_for_retrieval) != 10:
    raise ValueError("Expected 10 demo questions before evidence retrieval.")

retrieved_evidence_records = retrieve_evidence_for_demo_queries(query_log_rows_for_retrieval, module4_context_for_retrieval)
valid_statuses = {"success", "success_with_warnings", "failed"}

if len(retrieved_evidence_records) != 10:
    raise ValueError("There must be exactly 10 retrieved evidence records.")
if any(not record["retrieved_evidence"] for record in retrieved_evidence_records):
    raise ValueError("Every retrieved evidence record must be non-empty.")
if any(record["retrieval_status"] not in valid_statuses for record in retrieved_evidence_records):
    raise ValueError("Every retrieval_status must be success, success_with_warnings, or failed.")
if any(record["retrieval_status"] == "failed" for record in retrieved_evidence_records):
    raise ValueError("No retrieved evidence record should fail.")

retrieved_by_id = {record["query_id"]: record for record in retrieved_evidence_records}
q7_evidence = retrieved_by_id["Q7"]["retrieved_evidence"]
if q7_evidence.get("expected_combination_key") != "knn|vsn":
    raise ValueError("Q7 must retrieve expected_combination_key knn|vsn.")
if q7_evidence.get("combination_evidence", {}).get("classifier") != "knn":
    raise ValueError("Q7 combination evidence must be for classifier knn.")
if q7_evidence.get("combination_evidence", {}).get("normalization") != "vsn":
    raise ValueError("Q7 combination evidence must be for normalization vsn.")
if "overall_grounding_status" not in retrieved_by_id["Q8"]["retrieved_evidence"].get("grounding_summary", {}):
    raise ValueError("Q8 must include grounding_summary overall_grounding_status.")
if "example_high_delta_combination" not in retrieved_by_id["Q6"]["retrieved_evidence"]:
    raise ValueError("Q6 must include example_high_delta_combination.")
if "example_low_delta_combination" not in retrieved_by_id["Q6"]["retrieved_evidence"]:
    raise ValueError("Q6 must include example_low_delta_combination.")
if "most_stable_combinations" not in retrieved_by_id["Q4"]["retrieved_evidence"]:
    raise ValueError("Q4 must include most_stable_combinations.")
if "stable_combination_lookup_subset" not in retrieved_by_id["Q4"]["retrieved_evidence"]:
    raise ValueError("Q4 must include stable_combination_lookup_subset.")

save_retrieved_evidence(retrieved_evidence_records, OUTPUT_DIR / "module4_retrieved_evidence.json")

retrieval_display_rows = [
    {
        "query_id": record["query_id"],
        "route_subtype": record["route_subtype"],
        "retrieval_status": record["retrieval_status"],
        "number_of_retrieved_fields": len(record["retrieved_evidence"]),
        "retrieval_warnings": record["retrieval_warnings"],
    }
    for record in retrieved_evidence_records
]
retrieval_display = pd.DataFrame(retrieval_display_rows) if pd is not None else retrieval_display_rows

failed_retrievals = sum(1 for record in retrieved_evidence_records if record["retrieval_status"] == "failed")
warnings_count = sum(len(record["retrieval_warnings"]) for record in retrieved_evidence_records)
print("Module 4D retrieval validation summary")
print(f"retrieved_records: {len(retrieved_evidence_records)}")
print(f"failed_retrievals: {failed_retrievals}")
print(f"warnings_count: {warnings_count}")
print(f"q7_key: {q7_evidence.get('expected_combination_key')}")
print(f"q8_grounding_status: {retrieved_by_id['Q8']['retrieved_evidence']['grounding_summary'].get('overall_grounding_status')}")
retrieval_display

Module 4D retrieval validation summary
retrieved_records: 10
failed_retrievals: 0
warnings_count: 0
q7_key: knn|vsn
q8_grounding_status: pass_with_warnings


,query_id,route_subtype,retrieval_status,number_of_retrieved_fields,retrieval_warnings
0,Q1,overall_pattern,success,8,[]
1,Q2,batch_sensitive_classifier,success,7,[]
2,Q3,stable_normalization,success,6,[]
3,Q4,stable_combination_comparison,success,5,[]
4,Q5,split_50_vs_100,success,7,[]
5,Q6,delta_interpretation,success,8,[]
6,Q7,specific_combination,success,6,[]
7,Q8,grounding_consistency,success,4,[]
8,Q9,limitations,success,4,[]
9,Q10,report_ready_summary,success,12,[]


## Module 4E: Deterministic Template-Based Demo Chatbot Answer Generator

**Input:** `module4_outputs/module4_demo_context.json` and `module4_outputs/module4_retrieved_evidence.json`.

**Processing:** Generate deterministic template-based answers from retrieved evidence only, using the demo context only for global constraints and answer entity logging.

**Output:** Saved `module4_outputs/module4_chatbot_responses.json` plus a compact response validation table.

In [9]:
# ### module4 demo chatbot quality check cell 18
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: fmt_num
# Format numeric evidence consistently in chatbot answers.
def fmt_num(value, digits: int = 3) -> str:
    """Format a numeric value with fixed decimal places.

    Input: Numeric-like value and number of decimal digits.
    Output: Formatted string, or NA for missing/invalid values.
    """
    try:
        if value is None:
            return "NA"
        return f"{float(value):.{digits}f}"
    except (TypeError, ValueError):
        return "NA"


# ### Function: sectioned_answer
# Build a standardized answer with direct answer, evidence, interpretation, and limitation sections.
def sectioned_answer(direct: str, evidence: str, interpretation: str, limitation: str) -> str:
    """Combine text into the required four answer sections.

    Input: Text for direct answer, evidence, interpretation, and limitation sections.
    Output: Single answer string using exactly the required section headers.
    """
    return (
        f"Direct Answer:\n{direct.strip()}\n\n"
        f"Evidence Used:\n{evidence.strip()}\n\n"
        f"Interpretation:\n{interpretation.strip()}\n\n"
        f"Limitation / Caution:\n{limitation.strip()}"
    )


# ### Function: get_nested
# Safely retrieve nested values from context dictionaries.
def get_nested(d: dict, path: list[str], default=None):
    """Safely read a nested dictionary value.

    Input: Dictionary, list of nested keys, and optional default.
    Output: Nested value when present; otherwise the default.
    """
    current = d
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current


# ### Function: extract_answer_numeric_values
# Extract numeric values mentioned in an answer for quality checking.
def extract_answer_numeric_values(answer: str) -> list[str]:
    """Extract numeric strings from a generated answer.

    Input: Generated answer text.
    Output: List of numeric values appearing in the answer.
    """
    return re.findall(r"(?<![A-Za-z_])\d+(?:\.\d+)?", answer)


# ### Function: extract_answer_entities
# Extract classifier and normalization terms mentioned in an answer.
def extract_answer_entities(answer: str, context: dict) -> dict:
    """Extract known classifiers and normalizations from a generated answer.

    Input: Answer text and Module 4A context for known entity names.
    Output: Dictionary with classifier and normalization names used in the answer.
    """
    answer_norm = answer.lower()
    lookup = context.get("combination_lookup", {})
    classifiers = sorted({entry["classifier"] for entry in lookup.values() if re.search(rf"\b{re.escape(entry['classifier'])}\b", answer_norm)})
    normalizations = sorted({entry["normalization"] for entry in lookup.values() if re.search(rf"\b{re.escape(entry['normalization'])}\b", answer_norm)})
    return {"classifiers": classifiers, "normalizations": normalizations}


# ### Function: check_answer_sections_present
# Check whether a chatbot answer contains required sections.
def check_answer_sections_present(answer: str) -> dict:
    """Check that each required answer section header appears exactly once.

    Input: Generated answer text.
    Output: Dictionary mapping section header to exact-once presence.
    """
    headers = ["Direct Answer:", "Evidence Used:", "Interpretation:", "Limitation / Caution:"]
    return {header: answer.count(header) == 1 for header in headers}


# ### Function: _pair_key
# Build a stable key for one classifier-normalization pair.
def _pair_key(item: dict) -> str:
    """Format a classifier-normalization key from a combination dictionary.

    Input: Combination dictionary.
    Output: String key such as knn|vsn.
    """
    return f"{item.get('classifier')}|{item.get('normalization')}"


# ### Function: answer_overall_pattern
# Answer questions about the overall cross-batch trend.
def answer_overall_pattern(record: dict, context: dict) -> str:
    """Generate an overall-pattern answer from retrieved evidence.

    Input: Retrieved evidence record and context for logging only.
    Output: Sectioned deterministic answer string.
    """
    ev = record["retrieved_evidence"]
    counts = ev.get("overall_pattern_counts", {}).get("trend_label", {})
    increasing = counts.get("increasing", "NA")
    flat = counts.get("flat_or_weak", "NA")
    direct = f"Across {ev.get('scenario')}, most curves show increasing {ev.get('metric')} as split values move from split=50 toward split=100. The retrieved counts show {increasing} increasing curves and {flat} flat_or_weak curves."
    evidence = f"The answer uses split values {ev.get('split_values')}, overall_pattern_counts, and the dominant observation: {ev.get('dominant_observation')}"
    interpretation = "This pattern is consistent with reduced robustness under stronger batch-separated evaluation, because classification error is generally higher at larger split values."
    limitation = f"This describes observed performance patterns only. {ev.get('analysis_scope_note')} The answer uses only provided evidence and does not infer biological mechanisms."
    return sectioned_answer(direct, evidence, interpretation, limitation)


# ### Function: answer_batch_sensitive_classifier
# Answer which classifier appears most sensitive to batch shift.
def answer_batch_sensitive_classifier(record: dict, context: dict) -> str:
    """Generate a classifier-sensitivity comparison answer.

    Input: Retrieved evidence record and context for logging only.
    Output: Sectioned deterministic answer string.
    """
    ev = record["retrieved_evidence"]
    classifiers = ev.get("classifier_level_evidence", [])
    top_classifier = max(classifiers, key=lambda item: item.get("mean_delta_100_50", -1)) if classifiers else {}
    sensitive = ev.get("most_sensitive_combinations", [])[:3]
    sensitive_keys = ", ".join(_pair_key(item) for item in sensitive)
    direct = f"The classifier that appears most batch-sensitive is {top_classifier.get('classifier')}, based on the highest mean_delta_100_50 of {fmt_num(top_classifier.get('mean_delta_100_50'))} for {ev.get('metric')}."
    evidence = f"Classifier-level evidence reports {top_classifier.get('n_batch_sensitive')} batch-sensitive normalization settings for {top_classifier.get('classifier')}. The top sensitive combinations include {sensitive_keys}."
    interpretation = "This indicates that, in this scenario, the classifier's classification error changes more strongly between split=50 and split=100 than the other classifiers."
    limitation = "This comparison is scenario-specific and limited to the evaluated classifiers, normalizations, split values, and structured evidence. It does not support causal or biological conclusions."
    return sectioned_answer(direct, evidence, interpretation, limitation)


# ### Function: answer_stable_normalization
# Answer which normalization appears most stable.
def answer_stable_normalization(record: dict, context: dict) -> str:
    """Generate a normalization-stability comparison answer.

    Input: Retrieved evidence record and context for logging only.
    Output: Sectioned deterministic answer string.
    """
    ev = record["retrieved_evidence"]
    normalizations = ev.get("normalization_level_evidence", [])
    stable_norm = min(normalizations, key=lambda item: item.get("mean_delta_100_50", 999)) if normalizations else {}
    direct = f"The normalization method that appears most stable across classifiers is {stable_norm.get('normalization')}, with the lowest mean_delta_100_50 of {fmt_num(stable_norm.get('mean_delta_100_50'))} for {ev.get('metric')}."
    evidence = f"The retrieved normalization evidence reports dominant_robustness_flag={stable_norm.get('dominant_robustness_flag')} and {stable_norm.get('n_moderately_sensitive')} moderately_sensitive curves for {stable_norm.get('normalization')}."
    interpretation = "Here, stable means a smaller average change in classification error between split=50 and split=100 across the tested classifiers."
    limitation = f"This does not mean {stable_norm.get('normalization')} is universally best. {ev.get('analysis_scope_note')}"
    return sectioned_answer(direct, evidence, interpretation, limitation)


# ### Function: answer_stable_combination_comparison
# Compare stable classifier-normalization combinations.
def answer_stable_combination_comparison(record: dict, context: dict) -> str:
    """Generate a stable combination comparison answer.

    Input: Retrieved evidence record and context for logging only.
    Output: Sectioned deterministic answer string.
    """
    ev = record["retrieved_evidence"]
    top = ev.get("most_stable_combinations", [{}])[0]
    key = _pair_key(top)
    combo = ev.get("stable_combination_lookup_subset", {}).get(key, {})
    direct = f"The most stable classifier-normalization pair is {key}, with delta_100_50={fmt_num(top.get('delta_100_50'))} for {ev.get('metric')}."
    evidence = f"For {key}, error_50={fmt_num(combo.get('error_50', top.get('error_50')))} and error_100={fmt_num(combo.get('error_100', top.get('error_100')))}. The stable-combination ranking lists this pair first."
    interpretation = "In this answer, stable means a smaller change in classification error between split=50 and split=100, not necessarily the lowest absolute classification error."
    limitation = "This is a comparison within the current retrieved evidence only and should not be generalized beyond the tested scenario, classifiers, and normalizations."
    return sectioned_answer(direct, evidence, interpretation, limitation)


# ### Function: answer_split_50_vs_100
# Explain the meaning of split 50 versus split 100.
def answer_split_50_vs_100(record: dict, context: dict) -> str:
    """Generate a split interpretation answer.

    Input: Retrieved evidence record and context for logging only.
    Output: Sectioned deterministic answer string.
    """
    ev = record["retrieved_evidence"]
    sensitive = ev.get("most_sensitive_combinations", [{}])[0]
    stable = ev.get("most_stable_combinations", [{}])[0]
    direct = "split=50 represents a less batch-separated evaluation setting, while split=100 represents stronger batch-separated evaluation. In this experiment, many curves show higher classification error at split=100 than at split=50."
    evidence = f"The interpretation note says: {ev.get('interpretation_note')} A sensitive example is {_pair_key(sensitive)} with delta_100_50={fmt_num(sensitive.get('delta_100_50'))}; a stable example is {_pair_key(stable)} with delta_100_50={fmt_num(stable.get('delta_100_50'))}."
    interpretation = "The split comparison is therefore a robustness pattern: classification error often increases under the stronger batch-separated setting."
    limitation = "The split setting should not be described as causing the error increase. The answer is limited to observed classification error patterns in the retrieved evidence."
    return sectioned_answer(direct, evidence, interpretation, limitation)


# ### Function: answer_delta_interpretation
# Interpret delta and relative error increase metrics.
def answer_delta_interpretation(record: dict, context: dict) -> str:
    """Generate a delta_100_50 interpretation answer.

    Input: Retrieved evidence record and context for logging only.
    Output: Sectioned deterministic answer string.
    """
    ev = record["retrieved_evidence"]
    high = ev.get("example_high_delta_combination", {})
    low = ev.get("example_low_delta_combination", {})
    direct = "delta_100_50 is the difference between classification error at split=100 and classification error at split=50. A large positive delta_100_50 means classification error is higher at split=100."
    evidence = f"A high-delta example is {_pair_key(high)} with delta_100_50={fmt_num(high.get('delta_100_50'))}. A low-delta example is {_pair_key(low)} with delta_100_50={fmt_num(low.get('delta_100_50'))}."
    interpretation = "Because split=100 is the stronger batch-separated evaluation setting, a larger delta_100_50 is evidence of weaker robustness across those split settings."
    limitation = "This is a numeric performance interpretation only. It does not infer biological mechanisms or make claims outside the provided evidence."
    return sectioned_answer(direct, evidence, interpretation, limitation)


# ### Function: answer_specific_combination
# Answer questions about one specified classifier-normalization pair.
def answer_specific_combination(record: dict, context: dict) -> str:
    """Generate an exact combination answer.

    Input: Retrieved evidence record and context for logging only.
    Output: Sectioned deterministic answer string.
    """
    ev = record["retrieved_evidence"]
    combo = ev.get("combination_evidence", {})
    key = ev.get("expected_combination_key")
    direct = f"For {key}, the classifier is {combo.get('classifier')} and the normalization is {combo.get('normalization')}. The classification error rises from error_50={fmt_num(combo.get('error_50'))} at split=50 to error_100={fmt_num(combo.get('error_100'))} at split=100, with delta_100_50={fmt_num(combo.get('delta_100_50'))}."
    evidence = f"The retrieved record reports robustness_flag={combo.get('robustness_flag')}, degradation_type={combo.get('degradation_type')}, pattern_strength={combo.get('pattern_strength')}, and spike_type={combo.get('spike_type')}."
    interpretation = f"This supports interpreting {key} as batch-sensitive in this scenario, with a {combo.get('degradation_type')} pattern in classification error."
    limitation = "This answer uses only the exact retrieved combination evidence and does not compare against unrelated combinations or infer biological mechanisms."
    return sectioned_answer(direct, evidence, interpretation, limitation)


# ### Function: answer_grounding_consistency
# Explain grounding-check results for LLM output.
def answer_grounding_consistency(record: dict, context: dict) -> str:
    """Generate a grounding-consistency answer.

    Input: Retrieved evidence record and context for logging only.
    Output: Sectioned deterministic answer string.
    """
    ev = record["retrieved_evidence"]
    gs = ev.get("grounding_summary", {})
    status = gs.get("overall_grounding_status")
    classifier_cov = fmt_num(gs.get("classifier_coverage_rate"))
    norm_cov = fmt_num(gs.get("normalization_coverage_rate"))
    sens_cov = fmt_num(get_nested(gs, ["top_sensitive_pair_coverage", "coverage_rate"]))
    stable_cov = fmt_num(get_nested(gs, ["top_stable_pair_coverage", "coverage_rate"]))
    direct = f"The LLM-generated interpretation is broadly consistent with the observed evidence under this rule-based check, with overall status {status}."
    evidence = f"The grounding summary reports classifier coverage={classifier_cov}, normalization coverage={norm_cov}, top sensitive pair coverage={sens_cov}, and top stable pair coverage={stable_cov}."
    interpretation = "A pass_with_warnings result means the analysis is usable for a grounded demo, but some coverage checks may be incomplete."
    limitation = "This is not a full factual proof or benchmark-level evaluation. It is a lightweight rule-based grounding check over the retrieved structured evidence."
    return sectioned_answer(direct, evidence, interpretation, limitation)


# ### Function: answer_limitations
# Summarize project limitations using available evidence.
def answer_limitations(record: dict, context: dict) -> str:
    """Generate a limitation summary answer.

    Input: Retrieved evidence record and context for logging only.
    Output: Sectioned deterministic answer string.
    """
    ev = record["retrieved_evidence"]
    gs = ev.get("grounding_summary", {})
    direct = "The main limitations are that the analysis is restricted to the current scenario, tested classifiers, tested normalizations, split values, and classification error metric."
    evidence = f"The retrieved constraints are: {', '.join(ev.get('project_constraints', []))}. The grounding status is {gs.get('overall_grounding_status')}. The LLM limitation text states: {ev.get('llm_analysis_limitations')}"
    interpretation = "The evidence supports a performance-pattern summary, but not external conclusions or biological explanation. Coverage warnings should be treated as caution for demo use."
    limitation = f"{ev.get('analysis_scope_note')} This is a lightweight demo, not a full benchmark-level LLM evaluation."
    return sectioned_answer(direct, evidence, interpretation, limitation)


# ### Function: answer_report_ready_summary
# Generate a concise report-ready project summary.
def answer_report_ready_summary(record: dict, context: dict) -> str:
    """Generate a report-ready summary answer.

    Input: Retrieved evidence record and context for logging only.
    Output: Sectioned deterministic answer string.
    """
    ev = record["retrieved_evidence"]
    sensitive = ev.get("most_sensitive_combinations", [{}])[0]
    stable = ev.get("most_stable_combinations", [{}])[0]
    status = ev.get("grounding_summary", {}).get("overall_grounding_status")
    direct = "Report-ready summary: The dominant result is that classification error generally increases from split=50 to split=100, indicating reduced robustness under stronger batch-separated evaluation in this scenario."
    evidence = f"For scenario {ev.get('scenario')}, the top sensitive example is {_pair_key(sensitive)} with delta_100_50={fmt_num(sensitive.get('delta_100_50'))}, while the top stable example is {_pair_key(stable)} with delta_100_50={fmt_num(stable.get('delta_100_50'))}. The grounding status is {status}."
    interpretation = f"The structured evidence suggests that {_pair_key(sensitive)} is a high-sensitivity example and {_pair_key(stable)} is a stability example for the {ev.get('metric')} metric."
    limitation = f"The report-ready paragraph should be read as a summary of observed structured evidence only. {ev.get('analysis_scope_note')}"
    return sectioned_answer(direct, evidence, interpretation, limitation)


# ### Function: generate_answer
# Dispatch one routed query to the correct answer template.
def generate_answer(record: dict, context: dict) -> dict:
    """Generate one response object for a retrieved evidence record.

    Input: Retrieved evidence record and Module 4A context for logging metadata.
    Output: Response object with deterministic answer and validation metadata.
    """
    answer_functions = {
        "overall_pattern": answer_overall_pattern,
        "batch_sensitive_classifier": answer_batch_sensitive_classifier,
        "stable_normalization": answer_stable_normalization,
        "stable_combination_comparison": answer_stable_combination_comparison,
        "split_50_vs_100": answer_split_50_vs_100,
        "delta_interpretation": answer_delta_interpretation,
        "specific_combination": answer_specific_combination,
        "grounding_consistency": answer_grounding_consistency,
        "limitations": answer_limitations,
        "report_ready_summary": answer_report_ready_summary,
    }
    route_subtype = record.get("route_subtype")
    warnings = []
    if route_subtype not in answer_functions:
        answer = sectioned_answer(
            "No deterministic template is available for this route.",
            "No additional evidence was used.",
            "The query cannot be interpreted with the current template set.",
            "This response should not be used for reporting.",
        )
        warnings.append(f"Unsupported route_subtype: {route_subtype}")
    else:
        answer = answer_functions[route_subtype](record, context)

    sections_present = check_answer_sections_present(answer)
    return {
        "query_id": record.get("query_id"),
        "query": record.get("query"),
        "detected_query_type": record.get("detected_query_type"),
        "route_subtype": route_subtype,
        "answer": answer,
        "evidence_keys_used": list(record.get("retrieved_evidence", {}).keys()),
        "generation_mode": "template_based_grounded_demo",
        "answer_sections_present": sections_present,
        "answer_numeric_values_used": extract_answer_numeric_values(answer),
        "answer_entities_used": extract_answer_entities(answer, context),
        "answer_warnings": warnings,
    }


# ### Function: generate_demo_answers
# Generate answers for all retrieved demo-query records.
def generate_demo_answers(retrieved_records: list[dict], context: dict) -> list[dict]:
    """Generate deterministic response objects for all retrieved evidence records.

    Input: Retrieved evidence records and Module 4A context.
    Output: List of chatbot response objects.
    """
    return [generate_answer(record, context) for record in retrieved_records]


# ### Function: save_chatbot_responses
# Save chatbot responses as JSON for transcript and validation steps.
def save_chatbot_responses(responses: list[dict], output_path: Path) -> None:
    """Save generated chatbot responses as formatted JSON.

    Input: Response objects and target JSON path.
    Output: Writes response JSON and returns None.
    """
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(responses, file, indent=2, ensure_ascii=False)


module4_context_for_answers = load_json(MODULE4_DEMO_CONTEXT_PATH)
with (OUTPUT_DIR / "module4_retrieved_evidence.json").open("r", encoding="utf-8") as file:
    retrieved_records_for_answers = json.load(file)

module4_chatbot_responses = generate_demo_answers(retrieved_records_for_answers, module4_context_for_answers)
forbidden_phrases = ["proves", "caused by", "guarantees", "confirms biological mechanism", "biomarker discovery", "diagnostic marker"]

if len(module4_chatbot_responses) != 10:
    raise ValueError("There must be exactly 10 chatbot responses.")
if any(response["generation_mode"] != "template_based_grounded_demo" for response in module4_chatbot_responses):
    raise ValueError("All responses must use template_based_grounded_demo generation mode.")
if any(not all(response["answer_sections_present"].values()) for response in module4_chatbot_responses):
    raise ValueError("Every answer must include all four required section headers exactly once.")

forbidden_phrase_count = sum(
    1
    for response in module4_chatbot_responses
    for phrase in forbidden_phrases
    if phrase in response["answer"].lower()
)
if forbidden_phrase_count:
    raise ValueError("Generated answers contain forbidden phrases.")
if any(response["answer_warnings"] for response in module4_chatbot_responses):
    raise ValueError("No answer_warnings are expected for the fixed demo question set.")

responses_by_id = {response["query_id"]: response for response in module4_chatbot_responses}
q7_answer_norm = responses_by_id["Q7"]["answer"].lower()
q7_required_terms = ["knn", "vsn", "classification error", "split=50", "split=100", "delta_100_50", "robustness_flag", "degradation_type", "pattern_strength"]
q7_required_terms_ok = all(term in q7_answer_norm for term in q7_required_terms)
if not q7_required_terms_ok:
    raise ValueError("Q7 answer is missing required terms.")

q8_answer_norm = responses_by_id["Q8"]["answer"].lower()
q8_required_terms = ["pass_with_warnings", "classifier coverage", "normalization coverage"]
q8_required_terms_ok = all(term in q8_answer_norm for term in q8_required_terms)
if not q8_required_terms_ok:
    raise ValueError("Q8 answer is missing required terms.")

q10_answer_norm = responses_by_id["Q10"]["answer"].lower()
q10_required_terms = ["report-ready", "knn|vsn", "svm|non", "pass_with_warnings"]
q10_required_terms_ok = all(term in q10_answer_norm for term in q10_required_terms)
if not q10_required_terms_ok:
    raise ValueError("Q10 answer is missing required terms.")

save_chatbot_responses(module4_chatbot_responses, MODULE4_CHATBOT_RESPONSES_PATH)

response_display_rows = [
    {
        "query_id": response["query_id"],
        "route_subtype": response["route_subtype"],
        "generation_mode": response["generation_mode"],
        "sections_ok": all(response["answer_sections_present"].values()),
        "first_200_chars": response["answer"][:200],
    }
    for response in module4_chatbot_responses
]
response_display = pd.DataFrame(response_display_rows) if pd is not None else response_display_rows

all_sections_ok = all(all(response["answer_sections_present"].values()) for response in module4_chatbot_responses)
print("Module 4E answer generation validation summary")
print(f"response_count: {len(module4_chatbot_responses)}")
print(f"all_sections_ok: {all_sections_ok}")
print(f"forbidden_phrase_count: {forbidden_phrase_count}")
print(f"q7_required_terms_ok: {q7_required_terms_ok}")
print(f"q8_required_terms_ok: {q8_required_terms_ok}")
print(f"q10_required_terms_ok: {q10_required_terms_ok}")
response_display

Module 4E answer generation validation summary
response_count: 10
all_sections_ok: True
forbidden_phrase_count: 0
q7_required_terms_ok: True
q8_required_terms_ok: True
q10_required_terms_ok: True


,query_id,route_subtype,generation_mode,sections_ok,first_200_chars
0,Q1,overall_pattern,template_based_grounded_demo,True,"Direct Answer:\nAcross jama_scenario1_3, most ..."
1,Q2,batch_sensitive_classifier,template_based_grounded_demo,True,Direct Answer:\nThe classifier that appears mo...
2,Q3,stable_normalization,template_based_grounded_demo,True,Direct Answer:\nThe normalization method that ...
3,Q4,stable_combination_comparison,template_based_grounded_demo,True,Direct Answer:\nThe most stable classifier-nor...
4,Q5,split_50_vs_100,template_based_grounded_demo,True,Direct Answer:\nsplit=50 represents a less bat...
5,Q6,delta_interpretation,template_based_grounded_demo,True,Direct Answer:\ndelta_100_50 is the difference...
6,Q7,specific_combination,template_based_grounded_demo,True,"Direct Answer:\nFor knn|vsn, the classifier is..."
7,Q8,grounding_consistency,template_based_grounded_demo,True,Direct Answer:\nThe LLM-generated interpretati...
8,Q9,limitations,template_based_grounded_demo,True,Direct Answer:\nThe main limitations are that ...
9,Q10,report_ready_summary,template_based_grounded_demo,True,Direct Answer:\nReport-ready summary: The domi...


## Module 4F: Answer-Level Output Quality Checker

**Input:** `module4_outputs/module4_chatbot_responses.json`, `module4_outputs/module4_retrieved_evidence.json`, and `module4_outputs/module4_demo_context.json`.

**Processing:** Run deterministic answer-level quality checks for structure, forbidden claims, evidence terms, entity coverage, evidence-key alignment, limitation language, numeric grounding, and generation mode.

**Output:** Saved `module4_outputs/module4_answer_quality_log.csv` and `module4_outputs/module4_answer_quality_summary.json` plus a compact quality table.

In [10]:
# ### module4 demo chatbot quality check cell 20
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: normalize_text
# Normalize answer text before quality checking.
def normalize_text(text: str) -> str:
    """Normalize text for deterministic quality checks.

    Input: Raw answer text.
    Output: Lowercased text with normalized punctuation and spaces.
    """
    text_norm = text.lower().replace("-", "-")
    text_norm = re.sub(r"\s+", " ", text_norm)
    return text_norm.strip()


# ### Function: check_required_sections
# Verify that the analysis contains required report sections.
def check_required_sections(answer: str, response_metadata: dict | None = None) -> dict:
    """Check required answer sections from text and optional metadata.

    Input: Answer text and optional response metadata.
    Output: Dictionary with pass flag and section-level results.
    """
    headers = ["Direct Answer:", "Evidence Used:", "Interpretation:", "Limitation / Caution:"]
    text_results = {header: answer.count(header) == 1 for header in headers}
    metadata_results = response_metadata.get("answer_sections_present", {}) if response_metadata else {}
    metadata_passed = all(metadata_results.get(header, True) for header in headers)
    passed = all(text_results.values()) and metadata_passed
    return {"passed": passed, "text_results": text_results, "metadata_results": metadata_results}


# ### Function: is_negated_caution_context
# Define helper logic for is negated caution context used in this notebook step.
def is_negated_caution_context(text: str, phrase: str) -> bool:
    """Detect whether a risky phrase appears in a negated or cautionary context.

    Input: Normalized answer text and risky phrase.
    Output: True when nearby wording negates or cautions against the phrase.
    """
    phrase_norm = phrase.lower()
    caution_markers = ["do not", "does not", "no", "cannot", "not", "without", "doesn't", "avoid", "should not"]
    for match in re.finditer(re.escape(phrase_norm), text):
        start = max(0, match.start() - 80)
        end = min(len(text), match.end() + 80)
        window = text[start:end]
        if any(marker in window for marker in caution_markers):
            return True
    return False


# ### Function: check_forbidden_claims
# Define helper logic for check forbidden claims used in this notebook step.
def check_forbidden_claims(answer: str) -> dict:
    """Check for unsupported causal, biological, or overclaiming phrases.

    Input: Generated answer text.
    Output: Dictionary with pass flag and matched forbidden terms.
    """
    text = normalize_text(answer)
    forbidden_phrases = [
        "proves",
        "caused by",
        "guarantees",
        "confirms biological mechanism",
        "biomarker discovery",
        "diagnostic marker",
        "universal conclusion",
        "causal mechanism",
        "biological mechanism is confirmed",
        "disease mechanism",
        "clinical biomarker",
    ]
    matched = []
    for phrase in forbidden_phrases:
        if phrase in text:
            if "biological mechanism" in phrase and is_negated_caution_context(text, "biological mechanism"):
                continue
            matched.append(phrase)
    return {"passed": not matched, "matched_terms": matched}


# ### Function: required_terms_for_route
# Define helper logic for required terms for route used in this notebook step.
def required_terms_for_route(route_subtype: str) -> list[str]:
    """Return route-specific evidence terms required in an answer.

    Input: Route subtype string.
    Output: List of required answer terms.
    """
    terms_by_route = {
        "overall_pattern": ["classification error", "split", "increasing"],
        "batch_sensitive_classifier": ["classifier", "batch-sensitive", "delta_100_50"],
        "stable_normalization": ["normalization", "stable", "delta_100_50"],
        "stable_combination_comparison": ["classifier-normalization", "stable", "delta_100_50"],
        "split_50_vs_100": ["split=50", "split=100", "classification error"],
        "delta_interpretation": ["delta_100_50", "split=50", "split=100", "classification error"],
        "specific_combination": ["knn", "vsn", "classification error", "delta_100_50", "robustness_flag", "degradation_type", "pattern_strength"],
        "grounding_consistency": ["pass_with_warnings", "classifier coverage", "normalization coverage", "evidence"],
        "limitations": ["limitation", "scenario", "classifiers", "normalizations", "classification error"],
        "report_ready_summary": ["classification error", "split=50", "split=100", "knn|vsn", "svm|non", "pass_with_warnings"],
    }
    return terms_by_route.get(route_subtype, [])


# ### Function: check_evidence_terms
# Define helper logic for check evidence terms used in this notebook step.
def check_evidence_terms(answer: str, route_subtype: str) -> dict:
    """Check route-specific evidence terms in an answer.

    Input: Generated answer text and route subtype.
    Output: Dictionary with pass flag, present terms, and missing terms.
    """
    text = normalize_text(answer)
    required_terms = required_terms_for_route(route_subtype)
    present = [term for term in required_terms if term.lower() in text]
    missing = [term for term in required_terms if term.lower() not in text]
    return {"passed": not missing, "present_terms": present, "missing_terms": missing}


# ### Function: flatten_numeric_values
# Define helper logic for flatten numeric values used in this notebook step.
def flatten_numeric_values(obj, digits: int = 3) -> set[str]:
    """Recursively collect numeric values from retrieved evidence.

    Input: Nested object and rounding precision.
    Output: Set of numeric strings rounded to the requested precision plus integer forms.
    """
    values = set()
    if isinstance(obj, dict):
        for value in obj.values():
            values.update(flatten_numeric_values(value, digits))
    elif isinstance(obj, list):
        for value in obj:
            values.update(flatten_numeric_values(value, digits))
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool):
        values.add(f"{float(obj):.{digits}f}")
        if float(obj).is_integer():
            values.add(str(int(obj)))
    return values


# ### Function: check_numeric_grounding
# Define helper logic for check numeric grounding used in this notebook step.
def check_numeric_grounding(response: dict, retrieved_record: dict) -> dict:
    """Compare answer numeric values against retrieved evidence values.

    Input: Response object and corresponding retrieved evidence record.
    Output: Dictionary with pass flag, unmatched numbers, and matched numbers.
    """
    answer_values = set()
    for value in response.get("answer_numeric_values_used", []):
        try:
            number = float(value)
        except (TypeError, ValueError):
            continue
        answer_values.add(f"{number:.3f}")
        if number.is_integer():
            answer_values.add(str(int(number)))
    evidence_values = flatten_numeric_values(retrieved_record.get("retrieved_evidence", {}))
    if not answer_values:
        return {"passed": True, "matched_numbers": [], "unmatched_numbers": []}
    unmatched = sorted(value for value in answer_values if value not in evidence_values)
    matched = sorted(answer_values - set(unmatched))
    return {"passed": not unmatched, "matched_numbers": matched, "unmatched_numbers": unmatched}


# ### Function: check_entity_coverage
# Define helper logic for check entity coverage used in this notebook step.
def check_entity_coverage(response: dict, retrieved_record: dict, context: dict) -> dict:
    """Check route-specific expected entity coverage in an answer.

    Input: Response, retrieved evidence record, and context dictionary.
    Output: Dictionary with pass flag and missing entity notes.
    """
    route = response.get("route_subtype")
    answer = normalize_text(response.get("answer", ""))
    ev = retrieved_record.get("retrieved_evidence", {})
    missing = []

    if route == "specific_combination":
        combo = ev.get("combination_evidence", {})
        for term in [combo.get("classifier"), combo.get("normalization")]:
            if term and term not in answer:
                missing.append(term)
    elif route == "batch_sensitive_classifier":
        top = ev.get("most_sensitive_combinations", [{}])[0]
        top_classifier = top.get("classifier")
        top_pair = _pair_key(top)
        mentioned_classifiers = response.get("answer_entities_used", {}).get("classifiers", [])
        if not ((len(mentioned_classifiers) >= 2) or (top_classifier in answer and top_pair in answer)):
            missing.append("top classifier plus supporting combination or at least two classifiers")
    elif route == "stable_normalization":
        mentioned_normalizations = response.get("answer_entities_used", {}).get("normalizations", [])
        if not ((len(mentioned_normalizations) >= 2) or ("non" in answer and "normalization methods" in answer)):
            missing.append("non plus comparison against other normalization methods")
    elif route == "stable_combination_comparison":
        top = ev.get("most_stable_combinations", [{}])[0]
        top_pair = _pair_key(top)
        if top_pair not in answer:
            missing.append(top_pair)
    elif route == "split_50_vs_100":
        for term in ["split=50", "split=100"]:
            if term not in answer:
                missing.append(term)
    elif route == "delta_interpretation":
        high_pair = _pair_key(ev.get("example_high_delta_combination", {}))
        low_pair = _pair_key(ev.get("example_low_delta_combination", {}))
        for term in [high_pair, low_pair]:
            if term and term != "None|None" and term not in answer:
                missing.append(term)
    elif route == "grounding_consistency":
        if "pass_with_warnings" not in answer:
            missing.append("pass_with_warnings")
    elif route == "report_ready_summary":
        sensitive_pair = _pair_key(ev.get("most_sensitive_combinations", [{}])[0])
        stable_pair = _pair_key(ev.get("most_stable_combinations", [{}])[0])
        for term in [sensitive_pair, stable_pair]:
            if term and term != "None|None" and term not in answer:
                missing.append(term)

    return {"passed": not missing, "missing_entities": missing}


# ### Function: check_evidence_key_alignment
# Define helper logic for check evidence key alignment used in this notebook step.
def check_evidence_key_alignment(response: dict, retrieved_record: dict) -> dict:
    """Check alignment between response evidence keys and retrieved evidence keys.

    Input: Response object and retrieved evidence record.
    Output: Dictionary with pass flag, overlap, and incomplete keys.
    """
    used = set(response.get("evidence_keys_used", []))
    retrieved_context_keys = set(retrieved_record.get("retrieved_context_keys", []))
    retrieved_evidence_keys = set(retrieved_record.get("retrieved_evidence", {}).keys())
    allowed = retrieved_context_keys | retrieved_evidence_keys
    overlap = sorted(used & allowed)
    incomplete = sorted(used - allowed)
    return {"passed": bool(overlap), "overlap": overlap, "incomplete_keys": incomplete, "warning": bool(incomplete)}


# ### Function: check_limitation_present
# Define helper logic for check limitation present used in this notebook step.
def check_limitation_present(answer: str) -> dict:
    """Check whether limitation or caution language appears in an answer.

    Input: Generated answer text.
    Output: Dictionary with pass flag and matched terms.
    """
    text = normalize_text(answer)
    terms = ["limitation", "caution", "limited", "scenario-specific", "observed evidence", "observed pattern", "does not prove", "should not be interpreted as", "cannot infer", "not universal", "not causal", "only describes"]
    matched = [term for term in terms if term in text]
    return {"passed": bool(matched), "matched_terms": matched}


# ### Function: check_generation_mode
# Define helper logic for check generation mode used in this notebook step.
def check_generation_mode(response: dict) -> dict:
    """Verify deterministic template generation mode.

    Input: Response object.
    Output: Dictionary with pass flag and observed generation mode.
    """
    mode = response.get("generation_mode")
    return {"passed": mode == "template_based_grounded_demo", "generation_mode": mode}


# ### Function: check_single_answer_quality
# Define helper logic for check single answer quality used in this notebook step.
def check_single_answer_quality(response: dict, retrieved_record: dict, context: dict) -> dict:
    """Run all deterministic quality checks for one answer.

    Input: Response object, matching retrieved evidence record, and context.
    Output: Detailed quality log row.
    """
    answer = response.get("answer", "")
    checks = {
        "sections": check_required_sections(answer, response),
        "forbidden": check_forbidden_claims(answer),
        "terms": check_evidence_terms(answer, response.get("route_subtype", "")),
        "entities": check_entity_coverage(response, retrieved_record, context),
        "keys": check_evidence_key_alignment(response, retrieved_record),
        "limitation": check_limitation_present(answer),
        "numeric": check_numeric_grounding(response, retrieved_record),
        "mode": check_generation_mode(response),
    }
    warnings = []
    if not checks["terms"]["passed"]:
        warnings.append(f"missing_evidence_terms: {checks['terms']['missing_terms']}")
    if not checks["entities"]["passed"]:
        warnings.append(f"missing_entities: {checks['entities']['missing_entities']}")
    if checks["keys"].get("warning"):
        warnings.append(f"incomplete_evidence_key_alignment: {checks['keys']['incomplete_keys']}")
    if not checks["limitation"]["passed"]:
        warnings.append("limitation_language_missing_or_weak")
    if not checks["numeric"]["passed"]:
        warnings.append(f"ungrounded_numeric_values: {checks['numeric']['unmatched_numbers']}")

    fail_conditions = [
        not checks["sections"]["passed"],
        not checks["forbidden"]["passed"],
        not checks["mode"]["passed"],
        not checks["keys"]["passed"],
    ]
    if any(fail_conditions):
        quality_status = "fail"
    elif warnings:
        quality_status = "pass_with_warnings"
    else:
        quality_status = "pass"

    return {
        "query_id": response.get("query_id"),
        "query": response.get("query"),
        "detected_query_type": response.get("detected_query_type"),
        "route_subtype": response.get("route_subtype"),
        "generation_mode": response.get("generation_mode"),
        "required_sections_passed": checks["sections"]["passed"],
        "forbidden_claims_passed": checks["forbidden"]["passed"],
        "evidence_terms_passed": checks["terms"]["passed"],
        "entity_coverage_passed": checks["entities"]["passed"],
        "evidence_key_alignment_passed": checks["keys"]["passed"],
        "limitation_present": checks["limitation"]["passed"],
        "numeric_grounding_passed": checks["numeric"]["passed"],
        "warning_count": len(warnings),
        "warnings": warnings,
        "quality_status": quality_status,
    }


# ### Function: run_quality_checks
# Define helper logic for run quality checks used in this notebook step.
def run_quality_checks(responses: list[dict], retrieved_records: list[dict], context: dict) -> tuple["pd.DataFrame", dict]:
    """Run deterministic quality checks and build aggregate summary.

    Input: Response objects, retrieved evidence records, and context.
    Output: Quality log table and aggregate summary dictionary.
    """
    retrieved_by_id = {record["query_id"]: record for record in retrieved_records}
    rows = [check_single_answer_quality(response, retrieved_by_id[response["query_id"]], context) for response in responses]
    n_answers = len(rows)
    n_pass = sum(row["quality_status"] == "pass" for row in rows)
    n_pass_with_warnings = sum(row["quality_status"] == "pass_with_warnings" for row in rows)
    n_fail = sum(row["quality_status"] == "fail" for row in rows)
    warnings_by_type = {}
    for row in rows:
        for warning in row["warnings"]:
            warning_type = warning.split(":", 1)[0]
            warnings_by_type[warning_type] = warnings_by_type.get(warning_type, 0) + 1
    if n_fail > 0:
        overall_status = "fail"
    elif n_pass_with_warnings > 0:
        overall_status = "pass_with_warnings"
    else:
        overall_status = "pass"
    summary = {
        "n_answers": n_answers,
        "n_pass": n_pass,
        "n_pass_with_warnings": n_pass_with_warnings,
        "n_fail": n_fail,
        "pass_rate": n_pass / n_answers if n_answers else 0,
        "pass_or_warning_rate": (n_pass + n_pass_with_warnings) / n_answers if n_answers else 0,
        "warnings_by_type": warnings_by_type,
        "failed_query_ids": [row["query_id"] for row in rows if row["quality_status"] == "fail"],
        "warning_query_ids": [row["query_id"] for row in rows if row["warning_count"] > 0],
        "overall_module4_quality_status": overall_status,
        "generation_mode": "template_based_grounded_demo",
        "evaluated_files": [
            str(MODULE4_CHATBOT_RESPONSES_PATH),
            str(OUTPUT_DIR / "module4_retrieved_evidence.json"),
            str(MODULE4_DEMO_CONTEXT_PATH),
        ],
    }
    return (pd.DataFrame(rows) if pd is not None else rows), summary


# ### Function: save_quality_outputs
# Define helper logic for save quality outputs used in this notebook step.
def save_quality_outputs(quality_df, summary: dict, log_path: Path, summary_path: Path) -> None:
    """Save detailed and aggregate quality outputs.

    Input: Quality log table, summary dictionary, CSV path, and JSON path.
    Output: Writes CSV and JSON files.
    """
    rows = _table_to_rows(quality_df)
    fieldnames = [
        "query_id",
        "query",
        "detected_query_type",
        "route_subtype",
        "generation_mode",
        "required_sections_passed",
        "forbidden_claims_passed",
        "evidence_terms_passed",
        "entity_coverage_passed",
        "evidence_key_alignment_passed",
        "limitation_present",
        "numeric_grounding_passed",
        "warning_count",
        "warnings",
        "quality_status",
    ]
    with log_path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            csv_row = row.copy()
            csv_row["warnings"] = json.dumps(csv_row["warnings"], ensure_ascii=False, separators=(",", ":"))
            writer.writerow(csv_row)
    with summary_path.open("w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2, ensure_ascii=False)


module4_context_for_quality = load_json(MODULE4_DEMO_CONTEXT_PATH)
with MODULE4_CHATBOT_RESPONSES_PATH.open("r", encoding="utf-8") as file:
    responses_for_quality = json.load(file)
with (OUTPUT_DIR / "module4_retrieved_evidence.json").open("r", encoding="utf-8") as file:
    retrieved_records_for_quality = json.load(file)

module4_quality_log, module4_quality_summary = run_quality_checks(
    responses_for_quality,
    retrieved_records_for_quality,
    module4_context_for_quality,
)
quality_rows = _table_to_rows(module4_quality_log)

expected_quality_ids = [f"Q{index}" for index in range(1, 11)]
actual_quality_ids = [row["query_id"] for row in quality_rows]
if len(quality_rows) != 10:
    raise ValueError("There must be exactly 10 quality log rows.")
if actual_quality_ids != expected_quality_ids:
    raise ValueError("Quality log query IDs must be Q1 through Q10 in order.")
if any(row["quality_status"] == "fail" for row in quality_rows):
    raise ValueError("No response should have quality_status=fail.")
if not all(row["required_sections_passed"] for row in quality_rows):
    raise ValueError("required_sections_passed must be True for all responses.")
if not all(row["forbidden_claims_passed"] for row in quality_rows):
    raise ValueError("forbidden_claims_passed must be True for all responses.")
if not all(row["generation_mode"] == "template_based_grounded_demo" for row in quality_rows):
    raise ValueError("All responses must use template_based_grounded_demo.")
quality_by_id = {row["query_id"]: row for row in quality_rows}
if not quality_by_id["Q7"]["entity_coverage_passed"] or not quality_by_id["Q7"]["evidence_terms_passed"]:
    raise ValueError("Q7 must pass entity coverage and evidence term coverage.")
if not quality_by_id["Q8"]["evidence_terms_passed"]:
    raise ValueError("Q8 must pass grounding evidence term coverage.")
if not quality_by_id["Q10"]["evidence_terms_passed"]:
    raise ValueError("Q10 must pass report-ready evidence term coverage.")

save_quality_outputs(
    module4_quality_log,
    module4_quality_summary,
    MODULE4_ANSWER_QUALITY_LOG_PATH,
    MODULE4_ANSWER_QUALITY_SUMMARY_PATH,
)

if not MODULE4_ANSWER_QUALITY_LOG_PATH.exists() or not MODULE4_ANSWER_QUALITY_SUMMARY_PATH.exists():
    raise FileNotFoundError("Quality output files were not saved.")

quality_display_columns = [
    "query_id",
    "route_subtype",
    "quality_status",
    "warning_count",
    "required_sections_passed",
    "forbidden_claims_passed",
    "evidence_terms_passed",
    "entity_coverage_passed",
]
quality_display = module4_quality_log[quality_display_columns] if pd is not None else [{column: row[column] for column in quality_display_columns} for row in quality_rows]

print("Module 4F answer quality summary")
print(f"n_answers: {module4_quality_summary['n_answers']}")
print(f"n_pass: {module4_quality_summary['n_pass']}")
print(f"n_pass_with_warnings: {module4_quality_summary['n_pass_with_warnings']}")
print(f"n_fail: {module4_quality_summary['n_fail']}")
print(f"pass_rate: {fmt_num(module4_quality_summary['pass_rate'])}")
print(f"overall_module4_quality_status: {module4_quality_summary['overall_module4_quality_status']}")
print(f"failed_query_ids: {module4_quality_summary['failed_query_ids']}")
print(f"warning_query_ids: {module4_quality_summary['warning_query_ids']}")
quality_display

Module 4F answer quality summary
n_answers: 10
n_pass: 5
n_pass_with_warnings: 5
n_fail: 0
pass_rate: 0.500
overall_module4_quality_status: pass_with_warnings
failed_query_ids: []
warning_query_ids: ['Q3', 'Q4', 'Q5', 'Q6', 'Q7']


,query_id,route_subtype,quality_status,warning_count,required_sections_passed,forbidden_claims_passed,evidence_terms_passed,entity_coverage_passed
0,Q1,overall_pattern,pass,0,True,True,True,True
1,Q2,batch_sensitive_classifier,pass,0,True,True,True,True
2,Q3,stable_normalization,pass_with_warnings,1,True,True,True,False
3,Q4,stable_combination_comparison,pass_with_warnings,1,True,True,True,True
4,Q5,split_50_vs_100,pass_with_warnings,1,True,True,True,True
5,Q6,delta_interpretation,pass_with_warnings,1,True,True,True,True
6,Q7,specific_combination,pass_with_warnings,1,True,True,True,True
7,Q8,grounding_consistency,pass,0,True,True,True,True
8,Q9,limitations,pass,0,True,True,True,True
9,Q10,report_ready_summary,pass,0,True,True,True,True


## Module 4G: Export Demo Transcript for Final Report and Presentation

**Input:** Existing Module 4 outputs including context, responses, quality logs, routing logs, and retrieved evidence.

**Processing:** Merge response, route, retrieval, and quality metadata by query ID into a clean Markdown transcript for report or presentation use.

**Output:** Saved `module4_outputs/module4_demo_transcript.md` and a final Module 4 output existence checklist.

In [11]:
# ### module4 demo chatbot quality check cell 22
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: load_csv_fallback
# Load a CSV artifact with a fallback parser when pandas is unavailable.
def load_csv_fallback(path: Path):
    """Load CSV with pandas when available, otherwise with the csv module.

    Input: Path to a CSV file.
    Output: pandas DataFrame when functional, otherwise list of row dictionaries.
    """
    if pd is not None:
        return pd.read_csv(path)
    with path.open("r", encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


# ### Function: records_from_table
# Define helper logic for records from table used in this notebook step.
def records_from_table(table) -> list[dict]:
    """Convert a DataFrame-like table or list of dictionaries into records.

    Input: pandas DataFrame or list of dictionaries.
    Output: List of row dictionaries.
    """
    if pd is not None and hasattr(table, "to_dict"):
        return table.to_dict(orient="records")
    return list(table)


# ### Function: get_record_by_query_id
# Define helper logic for get record by query id used in this notebook step.
def get_record_by_query_id(records: list[dict], query_id: str) -> dict:
    """Find a record by query ID.

    Input: List of records and query ID string.
    Output: Matching record dictionary.
    """
    for record in records:
        if str(record.get("query_id")) == query_id:
            return record
    raise KeyError(f"No record found for query_id={query_id}")


# ### Function: as_compact_list
# Define helper logic for as compact list used in this notebook step.
def as_compact_list(value) -> str:
    """Convert list-like values into compact display text.

    Input: List, list-like string, or missing value.
    Output: Comma-separated display string or NA.
    """
    parsed = parse_list_cell(value)
    if parsed:
        return ", ".join(str(item) for item in parsed)
    if value is None or value == "" or str(value).lower() == "nan":
        return "NA"
    return str(value)


# ### Function: build_project_context_section
# Define helper logic for build project context section used in this notebook step.
def build_project_context_section(context: dict, responses: list[dict], quality_summary: dict) -> str:
    """Build the project context section for the transcript.

    Input: Module 4 context, responses, and quality summary.
    Output: Markdown project context section.
    """
    scenario_summary = context.get("scenario_summary", {})
    grounding_status = context.get("grounding_summary", {}).get("overall_grounding_status")
    generation_mode = quality_summary.get("generation_mode") or (responses[0].get("generation_mode") if responses else "NA")
    return "\n".join(
        [
            "## Project Context",
            "",
            f"- Scenario: {scenario_summary.get('scenario')}",
            f"- Metric: {scenario_summary.get('metric')}",
            f"- Split values: {scenario_summary.get('split_values')}",
            f"- Number of demo queries: {len(responses)}",
            f"- Number of classifier-normalization combinations: {scenario_summary.get('n_curves', len(context.get('combination_lookup', {})))}",
            f"- Module 3E grounding status: {grounding_status}",
            f"- Chatbot generation mode: {generation_mode}",
            "- Module 4 uses deterministic template-based grounded answers. No additional LLM API call is used for these answers.",
        ]
    )


# ### Function: build_quality_summary_section
# Define helper logic for build quality summary section used in this notebook step.
def build_quality_summary_section(quality_summary: dict) -> str:
    """Build the quality summary section for the transcript.

    Input: Module 4 answer quality summary dictionary.
    Output: Markdown quality summary section.
    """
    lines = [
        "## Quality Summary",
        "",
        f"- Number of demo queries: {quality_summary.get('n_answers')}",
        f"- Pass: {quality_summary.get('n_pass')}",
        f"- Pass with warnings: {quality_summary.get('n_pass_with_warnings')}",
        f"- Fail: {quality_summary.get('n_fail')}",
        f"- Pass rate: {fmt_num(quality_summary.get('pass_rate'))}",
        f"- Pass or warning rate: {fmt_num(quality_summary.get('pass_or_warning_rate'))}",
        f"- Overall Module 4 quality status: {quality_summary.get('overall_module4_quality_status')}",
    ]
    if quality_summary.get("n_fail") == 0:
        lines.append("- Note: warnings indicate strict coverage checks, not failed answers.")
    return "\n".join(lines)


# ### Function: build_single_qa_block
# Define helper logic for build single qa block used in this notebook step.
def build_single_qa_block(response: dict, quality_record: dict, route_record: dict, retrieved_record: dict) -> str:
    """Build one transcript Q&A block.

    Input: Response, quality, route, and retrieved evidence records for one query.
    Output: Markdown Q&A block.
    """
    query_id = response.get("query_id")
    return "\n".join(
        [
            f"### {query_id}. {response.get('query')}",
            "",
            f"**Detected query type:** {response.get('detected_query_type')}",
            f"**Route subtype:** {response.get('route_subtype')}",
            f"**Quality status:** {quality_record.get('quality_status')}",
            f"**Warning count:** {quality_record.get('warning_count')}",
            f"**Evidence keys used:** {as_compact_list(response.get('evidence_keys_used'))}",
            f"**Retrieved context keys:** {as_compact_list(route_record.get('retrieved_context_keys') or retrieved_record.get('retrieved_context_keys'))}",
            "",
            "**Chatbot answer:**",
            "",
            response.get("answer", ""),
            "",
            "---",
        ]
    )


# ### Function: build_demo_transcript
# Define helper logic for build demo transcript used in this notebook step.
def build_demo_transcript(
    responses: list[dict],
    quality_table,
    query_log_table,
    quality_summary: dict,
    context: dict,
    retrieved_records: list[dict],
) -> str:
    """Build the full demo transcript in Q1-Q10 order.

    Input: Responses, quality table, query log table, quality summary, context, and retrieved records.
    Output: Full Markdown transcript string.
    """
    quality_records = records_from_table(quality_table)
    route_records = records_from_table(query_log_table)
    response_order = sorted(responses, key=lambda item: int(item["query_id"].replace("Q", "")))
    blocks = [
        "# Module 4 Demo Chatbot Transcript",
        "",
        build_project_context_section(context, responses, quality_summary),
        "",
        build_quality_summary_section(quality_summary),
        "",
        "## Demo Q&A",
        "",
    ]
    for response in response_order:
        query_id = response["query_id"]
        blocks.append(
            build_single_qa_block(
                response=response,
                quality_record=get_record_by_query_id(quality_records, query_id),
                route_record=get_record_by_query_id(route_records, query_id),
                retrieved_record=get_record_by_query_id(retrieved_records, query_id),
            )
        )
    return "\n".join(blocks).strip() + "\n"


# ### Function: save_text
# Write report or prompt text to disk with a consistent encoding.
def save_text(text: str, path: Path) -> None:
    """Save text to disk with UTF-8 encoding.

    Input: Text string and output path.
    Output: Writes the text file and returns None.
    """
    path.write_text(text, encoding="utf-8")


# ### Function: build_module4_output_checklist
# Define helper logic for build module4 output checklist used in this notebook step.
def build_module4_output_checklist(output_dir: Path) -> list[dict]:
    """Build a checklist for expected Module 4 outputs.

    Input: Module 4 output directory.
    Output: List of dictionaries with file name, existence flag, and size in bytes.
    """
    expected_files = [
        "module4_demo_context.json",
        "module4_demo_questions.json",
        "module4_query_log.csv",
        "module4_retrieved_evidence.json",
        "module4_chatbot_responses.json",
        "module4_answer_quality_log.csv",
        "module4_answer_quality_summary.json",
        "module4_demo_transcript.md",
    ]
    rows = []
    for file_name in expected_files:
        path = output_dir / file_name
        rows.append({"file_name": file_name, "exists": path.exists(), "size_bytes": path.stat().st_size if path.exists() else 0})
    return rows


# ### Function: display_checklist
# Define helper logic for display checklist used in this notebook step.
def display_checklist(checklist_rows: list[dict]) -> None:
    """Display a compact output checklist.

    Input: Output checklist rows.
    Output: Displays a pandas table when available; otherwise prints compact rows.
    """
    if pd is not None:
        display(pd.DataFrame(checklist_rows))
    else:
        for row in checklist_rows:
            print(row)


module4_context_for_transcript = load_json(MODULE4_DEMO_CONTEXT_PATH)
with MODULE4_CHATBOT_RESPONSES_PATH.open("r", encoding="utf-8") as file:
    responses_for_transcript = json.load(file)
with MODULE4_ANSWER_QUALITY_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    quality_summary_for_transcript = json.load(file)
with (OUTPUT_DIR / "module4_retrieved_evidence.json").open("r", encoding="utf-8") as file:
    retrieved_records_for_transcript = json.load(file)

quality_table_for_transcript = load_csv_fallback(MODULE4_ANSWER_QUALITY_LOG_PATH)
query_log_table_for_transcript = load_csv_fallback(MODULE4_QUERY_LOG_PATH)

module4_demo_transcript = build_demo_transcript(
    responses=responses_for_transcript,
    quality_table=quality_table_for_transcript,
    query_log_table=query_log_table_for_transcript,
    quality_summary=quality_summary_for_transcript,
    context=module4_context_for_transcript,
    retrieved_records=retrieved_records_for_transcript,
)
save_text(module4_demo_transcript, MODULE4_DEMO_TRANSCRIPT_PATH)

transcript_exists = MODULE4_DEMO_TRANSCRIPT_PATH.exists()
transcript_size_bytes = MODULE4_DEMO_TRANSCRIPT_PATH.stat().st_size if transcript_exists else 0
if not transcript_exists or transcript_size_bytes <= 0:
    raise ValueError("Transcript file was not created or is empty.")
if "# Module 4 Demo Chatbot Transcript" not in module4_demo_transcript:
    raise ValueError("Transcript title is missing.")
for index in range(1, 11):
    if f"### Q{index}." not in module4_demo_transcript:
        raise ValueError(f"Transcript is missing Q{index} heading.")
required_transcript_terms = ["template_based_grounded_demo", "pass_with_warnings", "No additional LLM API"]
for term in required_transcript_terms:
    if term not in module4_demo_transcript:
        raise ValueError(f"Transcript is missing required term: {term}")
if module4_demo_transcript.count("**Quality status:**") != 10:
    raise ValueError("Transcript must include quality status for every query.")

module4_output_checklist = build_module4_output_checklist(OUTPUT_DIR)
all_expected_outputs_exist = all(row["exists"] for row in module4_output_checklist)
if not all_expected_outputs_exist:
    raise FileNotFoundError("One or more expected Module 4 output files are missing.")

print("Transcript preview (first 800 characters):")
print(module4_demo_transcript[:800])
display_checklist(module4_output_checklist)
print(f"transcript_exists: {transcript_exists}")
print(f"transcript_size_bytes: {transcript_size_bytes}")
print(f"all_expected_outputs_exist: {all_expected_outputs_exist}")

Transcript preview (first 800 characters):
# Module 4 Demo Chatbot Transcript

## Project Context

- Scenario: jama_scenario1_3
- Metric: classification error
- Split values: [50, 70, 80, 90, 100]
- Number of demo queries: 10
- Number of classifier-normalization combinations: 24
- Module 3E grounding status: pass_with_warnings
- Chatbot generation mode: template_based_grounded_demo
- Module 4 uses deterministic template-based grounded answers. No additional LLM API call is used for these answers.

## Quality Summary

- Number of demo queries: 10
- Pass: 5
- Pass with warnings: 5
- Fail: 0
- Pass rate: 0.500
- Pass or warning rate: 1.000
- Overall Module 4 quality status: pass_with_warnings
- Note: warnings indicate strict coverage checks, not failed answers.

## Demo Q&A

### Q1. What is the overall pattern observed across split va


,file_name,exists,size_bytes
0,module4_demo_context.json,True,49606
1,module4_demo_questions.json,True,7330
2,module4_query_log.csv,True,2654
3,module4_retrieved_evidence.json,True,62323
4,module4_chatbot_responses.json,True,17253
5,module4_answer_quality_log.csv,True,2389
6,module4_answer_quality_summary.json,True,623
7,module4_demo_transcript.md,True,13288


transcript_exists: True
transcript_size_bytes: 13288
all_expected_outputs_exist: True


## Module 4H: Final Validation and Cleanup

**Input:** All Module 4 generated output files in `module4_outputs`.

**Processing:** Run final read-only consistency validation over saved outputs, answer structure, quality status, transcript content, and expected file presence.

**Output:** Compact final checklist table with pass/fail status for each validation check.

In [12]:
# ### module4 demo chatbot quality check cell 24
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

# ### Function: safe_load_json
# Load optional validation artifacts without interrupting final cleanup.
def safe_load_json(path: Path):
    """Load a JSON file safely.

    Input: Path to a JSON file.
    Output: Parsed Python object.
    """
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


# ### Function: safe_load_csv_records
# Define helper logic for safe load csv records used in this notebook step.
def safe_load_csv_records(path: Path) -> list[dict]:
    """Load CSV records with the standard csv module.

    Input: Path to a CSV file.
    Output: List of row dictionaries.
    """
    with path.open("r", encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


# ### Function: has_required_sections
# Define helper logic for has required sections used in this notebook step.
def has_required_sections(answer: str) -> bool:
    """Check whether an answer contains all required section headers.

    Input: Chatbot answer text.
    Output: True when all four required section headers are present.
    """
    headers = ["Direct Answer:", "Evidence Used:", "Interpretation:", "Limitation / Caution:"]
    return all(header in answer for header in headers)


# ### Function: _check_row
# Define helper logic for check row used in this notebook step.
def _check_row(check_name: str, passed: bool, detail: str) -> dict:
    """Create one final validation checklist row.

    Input: Check name, pass flag, and detail text.
    Output: Checklist row dictionary.
    """
    return {"check_name": check_name, "status": "pass" if passed else "fail", "detail": detail}


# ### Function: validate_module4_outputs
# Define helper logic for validate module4 outputs used in this notebook step.
def validate_module4_outputs(output_dir: Path) -> list[dict]:
    """Run final read-only consistency validation over Module 4 outputs.

    Input: Module 4 output directory.
    Output: List of checklist rows with check_name, status, and detail.
    """
    rows = []
    expected_files = {
        "demo_context": output_dir / "module4_demo_context.json",
        "demo_questions": output_dir / "module4_demo_questions.json",
        "query_log": output_dir / "module4_query_log.csv",
        "retrieved_evidence": output_dir / "module4_retrieved_evidence.json",
        "chatbot_responses": output_dir / "module4_chatbot_responses.json",
        "answer_quality_log": output_dir / "module4_answer_quality_log.csv",
        "answer_quality_summary": output_dir / "module4_answer_quality_summary.json",
        "demo_transcript": output_dir / "module4_demo_transcript.md",
    }
    file_presence = {name: path.exists() for name, path in expected_files.items()}
    rows.append(_check_row("all_expected_output_files_exist", all(file_presence.values()), str(file_presence)))

    questions = safe_load_json(expected_files["demo_questions"]) if expected_files["demo_questions"].exists() else []
    expected_ids = [f"Q{index}" for index in range(1, 11)]
    question_ids = [question.get("query_id") for question in questions] if isinstance(questions, list) else []
    rows.append(_check_row("demo_questions_count", len(questions) == 10, f"count={len(questions)}"))
    rows.append(_check_row("demo_question_ids", question_ids == expected_ids, f"ids={question_ids}"))

    query_rows = safe_load_csv_records(expected_files["query_log"]) if expected_files["query_log"].exists() else []
    retrieved_records = safe_load_json(expected_files["retrieved_evidence"]) if expected_files["retrieved_evidence"].exists() else []
    responses = safe_load_json(expected_files["chatbot_responses"]) if expected_files["chatbot_responses"].exists() else []
    quality_rows = safe_load_csv_records(expected_files["answer_quality_log"]) if expected_files["answer_quality_log"].exists() else []
    quality_summary = safe_load_json(expected_files["answer_quality_summary"]) if expected_files["answer_quality_summary"].exists() else {}

    rows.append(_check_row("query_log_count", len(query_rows) == 10, f"count={len(query_rows)}"))
    rows.append(_check_row("retrieved_evidence_count", len(retrieved_records) == 10, f"count={len(retrieved_records)}"))
    rows.append(_check_row("chatbot_response_count", len(responses) == 10, f"count={len(responses)}"))
    rows.append(_check_row("answer_quality_row_count", len(quality_rows) == 10, f"count={len(quality_rows)}"))

    failed_quality_ids = [row.get("query_id") for row in quality_rows if row.get("quality_status") == "fail"]
    rows.append(_check_row("no_failed_answer_quality_rows", not failed_quality_ids, f"failed_query_ids={failed_quality_ids}"))

    section_failures = [response.get("query_id") for response in responses if not has_required_sections(response.get("answer", ""))]
    rows.append(_check_row("all_answers_have_required_sections", not section_failures, f"missing_sections={section_failures}"))

    modes = sorted({response.get("generation_mode") for response in responses})
    rows.append(_check_row("all_responses_template_based", modes == ["template_based_grounded_demo"], f"generation_modes={modes}"))

    summary_has_status = "overall_module4_quality_status" in quality_summary
    rows.append(_check_row("quality_summary_has_overall_status", summary_has_status, f"status={quality_summary.get('overall_module4_quality_status')}"))

    allowed_statuses = {"pass", "pass_with_warnings"}
    overall_status = quality_summary.get("overall_module4_quality_status")
    rows.append(_check_row("overall_quality_status_allowed", overall_status in allowed_statuses, f"status={overall_status}"))

    transcript_path = expected_files["demo_transcript"]
    transcript_text = transcript_path.read_text(encoding="utf-8") if transcript_path.exists() else ""
    rows.append(_check_row("transcript_exists_and_nonempty", transcript_path.exists() and transcript_path.stat().st_size > 0, f"size_bytes={transcript_path.stat().st_size if transcript_path.exists() else 0}"))
    transcript_checks = {
        "title": "# Module 4 Demo Chatbot Transcript" in transcript_text,
        "q1_to_q10": all(f"### Q{index}." in transcript_text for index in range(1, 11)),
        "generation_mode": "template_based_grounded_demo" in transcript_text,
        "quality_status": ("pass_with_warnings" in transcript_text or "pass" in transcript_text),
        "no_additional_llm_api": "No additional LLM API" in transcript_text,
    }
    rows.append(_check_row("transcript_required_content", all(transcript_checks.values()), str(transcript_checks)))
    rows.append(_check_row("all_module4_outputs_present_final", all(file_presence.values()), str(file_presence)))
    return rows


# ### Function: display_final_checklist
# Define helper logic for display final checklist used in this notebook step.
def display_final_checklist(checklist_rows: list[dict]) -> None:
    """Display the final validation checklist compactly.

    Input: Final validation checklist rows.
    Output: Displays a compact table or prints compact rows.
    """
    if pd is not None:
        display(pd.DataFrame(checklist_rows))
    else:
        for row in checklist_rows:
            print(row)


module4_final_checklist = validate_module4_outputs(OUTPUT_DIR)
display_final_checklist(module4_final_checklist)
total_checks = len(module4_final_checklist)
passed_checks = sum(row["status"] == "pass" for row in module4_final_checklist)
failed_checks = total_checks - passed_checks
module4_final_validation_status = "pass" if failed_checks == 0 else "fail"
print(f"total_checks: {total_checks}")
print(f"passed_checks: {passed_checks}")
print(f"failed_checks: {failed_checks}")
print(f"module4_final_validation_status: {module4_final_validation_status}")

,check_name,status,detail
0,all_expected_output_files_exist,pass,"{'demo_context': True, 'demo_questions': True,..."
1,demo_questions_count,pass,count=10
2,demo_question_ids,pass,"ids=['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7',..."
3,query_log_count,pass,count=10
4,retrieved_evidence_count,pass,count=10
5,chatbot_response_count,pass,count=10
6,answer_quality_row_count,pass,count=10
7,no_failed_answer_quality_rows,pass,failed_query_ids=[]
8,all_answers_have_required_sections,pass,missing_sections=[]
9,all_responses_template_based,pass,generation_modes=['template_based_grounded_demo']


total_checks: 15
passed_checks: 15
failed_checks: 0
module4_final_validation_status: pass


# Module 4 Completion Summary

Module 4 built a grounded demo chatbot using structured evidence from Modules 2 and 3. The chatbot supports scenario-level, classifier-level, normalization-level, classifier-normalization combination, split-interpretation, grounding/reliability, limitation, and report-ready summary questions. Each answer was generated deterministically from retrieved evidence using a template-based grounded demo mode, without additional LLM API calls in Module 4. Each answer was evaluated using rule-based answer-level quality checks for section completeness, evidence coverage, entity coverage, unsupported claims, numeric grounding, and limitation language. The module exports both machine-readable logs and a human-readable demo transcript for final report and presentation use.

- Number of demo queries: 10
- Number of chatbot responses: 10
- Answer quality status: pass_with_warnings with 0 failed answers
- Final transcript: `module4_outputs/module4_demo_transcript.md`

# Optional Appendix: API-Based Chatbot Mode

**Input:** Existing Module 4 context and retrieved evidence files.

**Processing:** Optional API-based answer generation from retrieved evidence only, using strict grounding constraints and the same four-section answer format.

**Output:** Separate API response and API quality files only when explicitly enabled.

**Default:** Disabled. No API calls are made unless `USE_API_CHATBOT` is manually set to `True` and the required environment/package configuration is available.

In [13]:
# ### module4 demo chatbot quality check cell 27
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

from google.colab import userdata
import os

secret_key = userdata.get("OPENAI_API_KEY")

print("Colab Secret found:", bool(secret_key))

if secret_key:
    os.environ["OPENAI_API_KEY"] = secret_key
    print("Environment variable set:", bool(os.getenv("OPENAI_API_KEY")))
else:
    print("Secret not found or notebook has no access to it.")

import os

print("OPENAI_API_KEY in env:", bool(os.getenv("OPENAI_API_KEY")))

try:
    from openai import OpenAI
    print("openai package available: True")
except Exception as e:
    print("openai package available: False")
    print(type(e).__name__, str(e))

Colab Secret found: True
Environment variable set: True
OPENAI_API_KEY in env: True
openai package available: True


In [15]:
# ### module4 demo chatbot quality check cell 28
# This cell is documented for repository readability; comments describe the purpose without changing execution logic.

import os

USE_API_CHATBOT = True
API_CHATBOT_MODEL = "gpt-4o-mini"
API_CHATBOT_TEMPERATURE = 0
API_CHATBOT_MAX_OUTPUT_TOKENS = 450
REQUIRED_API_SECTION_HEADERS = ["Direct Answer:", "Evidence Used:", "Interpretation:", "Limitation / Caution:"]

MODULE4_API_CHATBOT_RESPONSES_PATH = OUTPUT_DIR / "module4_api_chatbot_responses.json"
MODULE4_API_ANSWER_QUALITY_LOG_PATH = OUTPUT_DIR / "module4_api_answer_quality_log.csv"
MODULE4_API_ANSWER_QUALITY_SUMMARY_PATH = OUTPUT_DIR / "module4_api_answer_quality_summary.json"


# ### Function: missing_required_sections
# Define helper logic for missing required sections used in this notebook step.
def missing_required_sections(answer: str) -> list[str]:
    """Return required API answer section headers missing from an answer.

    Input: API answer text.
    Output: List of required section headers that are absent.
    """
    return [header for header in REQUIRED_API_SECTION_HEADERS if header not in answer]


# ### Function: build_api_prompt
# Define helper logic for build api prompt used in this notebook step.
def build_api_prompt(record: dict, context: dict) -> str:
    """Build a constrained API prompt from one retrieved evidence record.

    Input: Retrieved evidence record and context for global constraints.
    Output: Prompt string that restricts the model to retrieved evidence and required sections.
    """
    constraints = context.get("chatbot_constraints", [])
    prompt_payload = {
        "query_id": record.get("query_id"),
        "query": record.get("query"),
        "route_subtype": record.get("route_subtype"),
        "retrieved_evidence": record.get("retrieved_evidence", {}),
        "constraints": constraints,
    }
    return (
        "You are answering a question about structured ML robustness results.\n"
        "Use only the retrieved evidence below.\n"
        "Do not infer biological mechanisms.\n"
        "Do not claim causality.\n"
        "Do not introduce genes, biomarkers, external datasets, or external papers.\n"
        "If evidence is insufficient, say so explicitly.\n"
        "You must output exactly four sections and must include these exact headers verbatim. "
        "Do not use Markdown heading syntax. Do not rename, omit, or reorder the section headers.\n"
        "Required section headers, in this exact order:\n"
        "Direct Answer:\n"
        "Evidence Used:\n"
        "Interpretation:\n"
        "Limitation / Caution:\n\n"
        "The answer must mention classification error when discussing ML performance.\n"
        "The answer must mention split=50 and split=100 when discussing split effects or delta_100_50.\n"
        "The answer must use exact classifier and normalization names when the query asks about a specific combination.\n\n"
        f"Retrieved evidence payload:\n{json.dumps(prompt_payload, indent=2, ensure_ascii=False)}"
    )


# ### Function: api_mode_available
# Define helper logic for api mode available used in this notebook step.
def api_mode_available() -> tuple[bool, str]:
    """Check whether optional API mode can run.

    Input: Uses USE_API_CHATBOT, OPENAI_API_KEY environment variable, and import availability.
    Output: Tuple of availability flag and concise reason message.
    """
    if not USE_API_CHATBOT:
        return False, "Optional API chatbot mode is disabled. Existing template-based responses remain the official Module 4 output."
    if not os.environ.get("OPENAI_API_KEY"):
        return False, "USE_API_CHATBOT=True but OPENAI_API_KEY is missing; skipping API generation."
    try:
        from openai import OpenAI  # noqa: F401
    except ImportError:
        return False, "USE_API_CHATBOT=True but the OpenAI package is unavailable; skipping API generation."
    return True, "API mode is available."


# ### Function: call_api_chatbot
# Define helper logic for call api chatbot used in this notebook step.
def call_api_chatbot(prompt: str) -> str:
    """Call the optional API chatbot with deterministic settings.

    Input: Constrained prompt string.
    Output: API-generated answer text.
    """
    from openai import OpenAI

    client = OpenAI()
    response = client.responses.create(
        model=API_CHATBOT_MODEL,
        input=prompt,
        temperature=API_CHATBOT_TEMPERATURE,
        max_output_tokens=API_CHATBOT_MAX_OUTPUT_TOKENS,
    )
    return response.output_text


# ### Function: repair_api_answer_with_sections
# Define helper logic for repair api answer with sections used in this notebook step.
def repair_api_answer_with_sections(record: dict, invalid_answer: str, missing_sections: list[str], context: dict) -> str:
    """Retry an API answer that omitted one or more required section headers.

    Input: Retrieved evidence record, invalid answer text, missing headers, and context.
    Output: Rewritten API answer text.
    """
    repair_payload = {
        "query_id": record.get("query_id"),
        "query": record.get("query"),
        "route_subtype": record.get("route_subtype"),
        "retrieved_evidence": record.get("retrieved_evidence", {}),
        "constraints": context.get("chatbot_constraints", []),
        "missing_section_headers": missing_sections,
        "invalid_answer": invalid_answer,
    }
    repair_prompt = (
        "Rewrite the invalid answer using only the retrieved evidence.\n"
        "You must output exactly four sections and must include these exact headers verbatim. "
        "Do not use Markdown heading syntax. Do not rename, omit, or reorder the section headers.\n"
        "Required section headers, in this exact order:\n"
        "Direct Answer:\n"
        "Evidence Used:\n"
        "Interpretation:\n"
        "Limitation / Caution:\n\n"
        "Do not infer biological mechanisms. Do not claim causality. Do not introduce external evidence.\n\n"
        f"Repair payload:\n{json.dumps(repair_payload, indent=2, ensure_ascii=False)}"
    )
    return call_api_chatbot(repair_prompt)


# ### Function: _api_response_object
# Define helper logic for api response object used in this notebook step.
def _api_response_object(record: dict, answer: str, context: dict, status: str, missing_sections: list[str], error_message: str = "") -> dict:
    """Build a response object for optional API mode.

    Input: Retrieved record, answer text, context, status, missing headers, and optional error message.
    Output: API response object using the template response schema plus API diagnostic fields.
    """
    return {
        "query_id": record.get("query_id"),
        "query": record.get("query"),
        "detected_query_type": record.get("detected_query_type"),
        "route_subtype": record.get("route_subtype"),
        "answer": answer,
        "evidence_keys_used": list(record.get("retrieved_evidence", {}).keys()),
        "generation_mode": "api_grounded_demo_optional",
        "answer_sections_present": check_answer_sections_present(answer),
        "answer_numeric_values_used": extract_answer_numeric_values(answer),
        "answer_entities_used": extract_answer_entities(answer, context),
        "answer_warnings": [f"missing_sections: {missing_sections}"] if missing_sections else [],
        "api_generation_status": status,
        "missing_sections": missing_sections,
        "api_error_message": error_message,
    }


# ### Function: generate_api_answer
# Generate an API-based answer constrained by retrieved evidence.
def generate_api_answer(record: dict, context: dict) -> dict:
    """Generate one optional API-based response object with one formatting retry.

    Input: Retrieved evidence record and context for global constraints/entity logging.
    Output: API response object with robust API status fields.
    """
    try:
        answer = call_api_chatbot(build_api_prompt(record, context))
        missing_sections = missing_required_sections(answer)
        if not missing_sections:
            return _api_response_object(record, answer, context, "success", [])

        repaired_answer = repair_api_answer_with_sections(record, answer, missing_sections, context)
        repaired_missing_sections = missing_required_sections(repaired_answer)
        if not repaired_missing_sections:
            return _api_response_object(record, repaired_answer, context, "success_after_retry", [])
        return _api_response_object(record, repaired_answer, context, "format_failed", repaired_missing_sections)
    except Exception as exc:
        return _api_response_object(record, "", context, "api_error", REQUIRED_API_SECTION_HEADERS, f"{type(exc).__name__}: {exc}")


# ### Function: generate_api_demo_answers
# Define helper logic for generate api demo answers used in this notebook step.
def generate_api_demo_answers(retrieved_records: list[dict], context: dict) -> list[dict]:
    """Generate optional API answers for all fixed retrieved evidence records.

    Input: Retrieved evidence records and context.
    Output: List of API response objects, including diagnostic failures.
    """
    return [generate_api_answer(record, context) for record in retrieved_records]


# ### Function: save_api_chatbot_responses
# Define helper logic for save api chatbot responses used in this notebook step.
def save_api_chatbot_responses(responses: list[dict], output_path: Path) -> None:
    """Save optional API chatbot responses as JSON.

    Input: API response objects and output path.
    Output: Writes JSON response file.
    """
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(responses, file, indent=2, ensure_ascii=False)


# ### Function: run_api_quality_checks_if_available
# Define helper logic for run api quality checks if available used in this notebook step.
def run_api_quality_checks_if_available(api_responses: list[dict], retrieved_records: list[dict], context: dict) -> tuple:
    """Run API response quality checks by reusing Module 4F functions.

    Input: Successful API responses, matching retrieved evidence records, and context.
    Output: Tuple of quality log table and summary, or (None, None) when unavailable.
    """
    if "run_quality_checks" not in globals() or "save_quality_outputs" not in globals():
        return None, None
    if not api_responses:
        return None, None
    quality_df, quality_summary = run_quality_checks(api_responses, retrieved_records, context)
    save_quality_outputs(
        quality_df,
        quality_summary,
        MODULE4_API_ANSWER_QUALITY_LOG_PATH,
        MODULE4_API_ANSWER_QUALITY_SUMMARY_PATH,
    )
    return quality_df, quality_summary


# ### Function: _print_api_summary
# Define helper logic for print api summary used in this notebook step.
def _print_api_summary(
    api_mode_enabled: bool,
    api_generation_ran: bool,
    api_total_records: int = 0,
    api_success_count: int = 0,
    api_success_after_retry_count: int = 0,
    api_format_failed_count: int = 0,
    api_error_count: int = 0,
    api_quality_checked_count: int = 0,
    api_output_files_created: list[str] | None = None,
) -> None:
    """Print the compact optional API mode summary.

    Input: API mode counters and output file list.
    Output: Compact printed status only.
    """
    print(f"api_mode_enabled: {api_mode_enabled}")
    print(f"api_generation_ran: {api_generation_ran}")
    print(f"api_total_records: {api_total_records}")
    print(f"api_success_count: {api_success_count}")
    print(f"api_success_after_retry_count: {api_success_after_retry_count}")
    print(f"api_format_failed_count: {api_format_failed_count}")
    print(f"api_error_count: {api_error_count}")
    print(f"api_quality_checked_count: {api_quality_checked_count}")
    print(f"api_output_files_created: {api_output_files_created or []}")
    print("official_module4_output: template_based_grounded_demo")


# ### Function: run_optional_api_chatbot_section
# Define helper logic for run optional api chatbot section used in this notebook step.
def run_optional_api_chatbot_section() -> None:
    """Run or skip the optional API chatbot appendix section.

    Input: Uses saved Module 4 context and retrieved evidence files.
    Output: Prints compact status; writes API-only files only when enabled and at least one answer succeeds.
    """
    available, reason = api_mode_available()
    if not available:
        _print_api_summary(api_mode_enabled=USE_API_CHATBOT, api_generation_ran=False)
        print(f"api_mode_status: {reason}")
        return

    context = load_json(MODULE4_DEMO_CONTEXT_PATH)
    with (OUTPUT_DIR / "module4_retrieved_evidence.json").open("r", encoding="utf-8") as file:
        retrieved_records = json.load(file)

    api_responses = generate_api_demo_answers(retrieved_records, context)
    success_statuses = {"success", "success_after_retry"}
    successful_responses = [response for response in api_responses if response.get("api_generation_status") in success_statuses]
    successful_ids = {response.get("query_id") for response in successful_responses}
    retrieved_records_for_quality = [record for record in retrieved_records if record.get("query_id") in successful_ids]

    api_success_count = sum(response.get("api_generation_status") == "success" for response in api_responses)
    api_success_after_retry_count = sum(response.get("api_generation_status") == "success_after_retry" for response in api_responses)
    api_format_failed_count = sum(response.get("api_generation_status") == "format_failed" for response in api_responses)
    api_error_count = sum(response.get("api_generation_status") == "api_error" for response in api_responses)
    api_output_files_created = []
    api_quality_checked_count = 0

    if successful_responses:
        save_api_chatbot_responses(api_responses, MODULE4_API_CHATBOT_RESPONSES_PATH)
        api_output_files_created.append(str(MODULE4_API_CHATBOT_RESPONSES_PATH))
        _, api_quality_summary = run_api_quality_checks_if_available(successful_responses, retrieved_records_for_quality, context)
        api_quality_checked_count = len(successful_responses) if api_quality_summary is not None else 0
        if api_quality_summary is not None:
            api_output_files_created.extend([str(MODULE4_API_ANSWER_QUALITY_LOG_PATH), str(MODULE4_API_ANSWER_QUALITY_SUMMARY_PATH)])

    _print_api_summary(
        api_mode_enabled=True,
        api_generation_ran=True,
        api_total_records=len(api_responses),
        api_success_count=api_success_count,
        api_success_after_retry_count=api_success_after_retry_count,
        api_format_failed_count=api_format_failed_count,
        api_error_count=api_error_count,
        api_quality_checked_count=api_quality_checked_count,
        api_output_files_created=api_output_files_created,
    )


run_optional_api_chatbot_section()

api_mode_enabled: True
api_generation_ran: True
api_total_records: 10
api_success_count: 10
api_success_after_retry_count: 0
api_format_failed_count: 0
api_error_count: 0
api_quality_checked_count: 10
api_output_files_created: ['module4_outputs/module4_api_chatbot_responses.json', 'module4_outputs/module4_api_answer_quality_log.csv', 'module4_outputs/module4_api_answer_quality_summary.json']
official_module4_output: template_based_grounded_demo


This optional API mode is not part of the official validated Module 4 pipeline unless explicitly enabled and executed. The official Module 4 results remain the deterministic template-based grounded demo outputs that passed final validation.